In [ ]:
import os
# Configure before running. See README; use a fresh output tree.
os.environ.setdefault("EGLIMUC_WORKSPACE", "/content/eglimuc")
os.environ.setdefault("EGLIMUC_MOUNT_DRIVE", "0")


In [ ]:
# ============================================================
# 00_legacy_manuscript_reproducibility_audit.ipynb
# COMPLETE SINGLE-CELL CODE
# ============================================================
#
# PURPOSE
# -------
# Audit the preserved pre-Route-A analyses against numerical
# and methodological claims in the previously submitted paper.
#
# This notebook separates:
#
#   1. Computational reproducibility
#      - Do saved legacy prediction files reproduce the
#        manuscript metrics?
#
#   2. Reporting accuracy
#      - Were values copied correctly into the paper?
#
#   3. Inferential validity
#      - Were development and official-test results used
#        appropriately for selection and final evaluation?
#
# IMPORTANT
# ---------
# This notebook:
#   - performs NO training
#   - loads NO model checkpoints
#   - runs NO image inference
#   - does NOT access raw development or raw test images
#   - does NOT modify legacy files
#
# It reads only saved:
#   - CSV prediction files
#   - JSON configuration files
#   - TXT/MD summaries
#   - notebook/script source text for provenance
#
# The authoritative revised paper remains Route A 01–09.
# Legacy analyses are retained for provenance and auditing only.
#
# MAIN OUTPUTS
# ------------
# legacy_artifact_inventory.csv
# legacy_prediction_file_catalog.csv
# legacy_config_inventory.csv
# legacy_metrics_recomputed.csv
# legacy_patient_prediction_consistency.csv
# legacy_fold_patient_counts.csv
# legacy_confusion_metrics_recomputed.csv
# legacy_aggregation_saved_summary.csv
# legacy_aggregation_recomputed.csv
# legacy_claim_registry.csv
# legacy_manuscript_value_comparison.csv
# legacy_discrepancy_report.csv
# legacy_inferential_validity_audit.csv
# legacy_code_evidence.csv
# legacy_provenance_manifest.json
# LEGACY_AUDIT_SUMMARY.txt
# ============================================================


# ============================================================
# 0. Install only the analysis dependency that may be missing
# ============================================================

%pip -q install scikit-learn==1.6.1


# ============================================================
# 1. Imports
# ============================================================

import json
import math
import os
import re
import time
import hashlib
import platform
import warnings

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

from scipy.stats import trim_mean

from sklearn.metrics import (
    cohen_kappa_score,
    mean_absolute_error,
    f1_score,
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from IPython.display import display


warnings.filterwarnings("ignore")


# ============================================================
# 2. Mount Google Drive
# ============================================================

from google.colab import drive

(drive.mount(
    "/content/drive",
    force_remount=True,
) if os.environ.get("EGLIMUC_MOUNT_DRIVE", "0") == "1" else None)


# ============================================================
# 3. Paths and read-only policy
# ============================================================

LEGACY_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT')
)

ROUTE_A_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026')
)

OUTPUT_ROOT = (
    ROUTE_A_ROOT
    / "00_legacy_manuscript_reproducibility_audit"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


if not LEGACY_ROOT.exists():
    raise FileNotFoundError(
        f"Legacy analysis root was not found:\n{LEGACY_ROOT}"
    )


# Raw image roots are recorded only as forbidden locations.
# This notebook never reads them.
FORBIDDEN_RAW_IMAGE_ROOTS = [
    Path(
        (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/train_and_validation_sets')
    ),

    Path(
        (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/test_set')
    ),
]


LOAD_MODEL_CHECKPOINTS = False
RUN_MODEL_INFERENCE = False
ACCESS_RAW_IMAGES = False


if LOAD_MODEL_CHECKPOINTS:
    raise RuntimeError(
        "Checkpoint loading must remain disabled."
    )


if RUN_MODEL_INFERENCE:
    raise RuntimeError(
        "Model inference must remain disabled."
    )


if ACCESS_RAW_IMAGES:
    raise RuntimeError(
        "Raw-image access must remain disabled."
    )


print("=" * 125)
print("LEGACY REPRODUCIBILITY AUDIT — READ-ONLY MODE")
print("=" * 125)

print(
    "Legacy root:",
    LEGACY_ROOT,
)

print(
    "Audit output:",
    OUTPUT_ROOT,
)

print(
    "Model checkpoints loaded:",
    False,
)

print(
    "Image inference performed:",
    False,
)

print(
    "Raw image folders accessed:",
    False,
)


# ============================================================
# 4. Confirmed legacy experiment directories
# ============================================================

LEGACY_EXPERIMENTS = {
    "M1_vit_oldordmae": {
        "manuscript_label": "M1",

        "display_name": (
            "ViT + expected-value ordinal MAE"
        ),

        "path": (
            LEGACY_ROOT
            / "ViT_loss_study"
            / "vit_L2_ce_oldordmae_w010"
        ),
    },

    "VIT_L4_exploratory": {
        "manuscript_label": None,

        "display_name": (
            "Legacy exploratory ViT ordinal distance"
        ),

        "path": (
            LEGACY_ROOT
            / "ViT_loss_study"
            / "vit_L4_orddist_a2_w010"
        ),
    },

    "M2_swin_oldordmae": {
        "manuscript_label": "M2",

        "display_name": (
            "Swin + expected-value ordinal MAE"
        ),

        "path": (
            LEGACY_ROOT
            / "Swin_backbone_study"
            / "swin_oldordmae_w010"
        ),
    },

    "SWIN_A1W020_exploratory": {
        "manuscript_label": None,

        "display_name": (
            "Legacy exploratory Swin "
            "ordinal distance alpha=1 lambda=.20"
        ),

        "path": (
            LEGACY_ROOT
            / "Swin_backbone_study"
            / "swin_orddist_a1_w020"
        ),
    },

    "M3_swin_orddist_a2": {
        "manuscript_label": "M3",

        "display_name": (
            "Swin + expected ordinal distance "
            "alpha=2 lambda=.10"
        ),

        "path": (
            LEGACY_ROOT
            / "Swin_backbone_study"
            / "swin_orddist_a2_w010"
        ),
    },

    "M4_convnext_orddist_a2": {
        "manuscript_label": "M4",

        "display_name": (
            "ConvNeXt + expected ordinal distance "
            "alpha=2 lambda=.10"
        ),

        "path": (
            LEGACY_ROOT
            / "ConvNeXt_backbone_study"
            / "convnext_orddist_a2_w010"
        ),
    },

    "M5_maxvit_orddist_a2": {
        "manuscript_label": "M5",

        "display_name": (
            "MaxViT + expected ordinal distance "
            "alpha=2 lambda=.10"
        ),

        "path": (
            LEGACY_ROOT
            / "MaxViT_backbone_study"
            / "maxvit_orddist_a2_w010"
        ),
    },

    "A0_default": {
        "manuscript_label": "A0_default",

        "display_name": (
            "Legacy Swin default ablation"
        ),

        "path": (
            LEGACY_ROOT
            / "Swin_backbone_study_ablation"
            / "swin_orddist_a2_w010__A0_default"
        ),
    },

    "A1_no_tta": {
        "manuscript_label": "A1_no_tta",

        "display_name": (
            "Legacy Swin no-TTA ablation"
        ),

        "path": (
            LEGACY_ROOT
            / "Swin_backbone_study_ablation"
            / "swin_orddist_a2_w010__A1_no_tta"
        ),
    },

    "A2_reduced_aug": {
        "manuscript_label": "A2_reduced_aug",

        "display_name": (
            "Legacy Swin reduced-augmentation ablation"
        ),

        "path": (
            LEGACY_ROOT
            / "Swin_backbone_study_ablation"
            / "swin_orddist_a2_w010__A2_reduced_aug"
        ),
    },

    "A3_no_class_weights": {
        "manuscript_label": "A3_no_class_weights",

        "display_name": (
            "Legacy Swin no-class-weight ablation"
        ),

        "path": (
            LEGACY_ROOT
            / "Swin_backbone_study_ablation"
            / "swin_orddist_a2_w010__A3_no_class_weights"
        ),
    },
}


AGGREGATION_ROOT = (
    LEGACY_EXPERIMENTS[
        "A3_no_class_weights"
    ][
        "path"
    ]
    / "aggregation_sensitivity"
)


CODE_ROOT = (
    LEGACY_ROOT
    / "Code"
)


ENSEMBLE_ROOT = (
    LEGACY_ROOT
    / "Ensemble_study"
)


missing_experiment_directories = [
    str(
        experiment_info[
            "path"
        ]
    )

    for experiment_info in (
        LEGACY_EXPERIMENTS.values()
    )

    if not experiment_info[
        "path"
    ].exists()
]


if missing_experiment_directories:
    print(
        "\nWARNING: Some expected legacy directories "
        "were not found:"
    )

    for path in missing_experiment_directories:
        print(
            " -",
            path,
        )


print(
    "\nAggregation root exists:",
    AGGREGATION_ROOT.exists(),
)

print(
    "Legacy code root exists:",
    CODE_ROOT.exists(),
)

print(
    "Legacy ensemble root exists:",
    ENSEMBLE_ROOT.exists(),
)


# ============================================================
# 5. Atomic output and safe-read helpers
# ============================================================

MAX_READ_RETRIES = 5
INITIAL_RETRY_DELAY_SECONDS = 1.0


def temporary_path(
    path,
):
    path = Path(
        path
    )

    return path.with_name(
        path.name
        +
        ".temporary"
    )


def atomic_csv_save(
    dataframe,
    path,
    index=False,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = temporary_path(
        path
    )

    if temp_path.exists():
        temp_path.unlink()

    dataframe.to_csv(
        temp_path,
        index=index,
    )

    os.replace(
        temp_path,
        path,
    )


def atomic_json_save(
    payload,
    path,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = temporary_path(
        path
    )

    if temp_path.exists():
        temp_path.unlink()

    with open(
        temp_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            payload,
            file,
            indent=2,
            default=str,
        )

        file.flush()

        os.fsync(
            file.fileno()
        )

    os.replace(
        temp_path,
        path,
    )


def atomic_text_save(
    text,
    path,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = temporary_path(
        path
    )

    if temp_path.exists():
        temp_path.unlink()

    with open(
        temp_path,
        "w",
        encoding="utf-8",
    ) as file:
        file.write(
            text
        )

        file.flush()

        os.fsync(
            file.fileno()
        )

    os.replace(
        temp_path,
        path,
    )


def safe_read_csv(
    path,
):
    path = Path(
        path
    )

    last_error = None

    for attempt in range(
        MAX_READ_RETRIES
    ):
        try:
            return pd.read_csv(
                path,
                low_memory=False,
            )

        except Exception as error:
            last_error = error

            if (
                attempt
                ==
                MAX_READ_RETRIES - 1
            ):
                break

            time.sleep(
                INITIAL_RETRY_DELAY_SECONDS
                *
                (
                    attempt + 1
                )
            )

    raise RuntimeError(
        f"Could not read CSV:\n{path}\n"
        f"Last error: {repr(last_error)}"
    )


def safe_read_json(
    path,
):
    path = Path(
        path
    )

    last_error = None

    for attempt in range(
        MAX_READ_RETRIES
    ):
        try:
            with open(
                path,
                "r",
                encoding="utf-8",
            ) as file:
                return json.load(
                    file
                )

        except Exception as error:
            last_error = error

            if (
                attempt
                ==
                MAX_READ_RETRIES - 1
            ):
                break

            time.sleep(
                INITIAL_RETRY_DELAY_SECONDS
                *
                (
                    attempt + 1
                )
            )

    raise RuntimeError(
        f"Could not read JSON:\n{path}\n"
        f"Last error: {repr(last_error)}"
    )


def safe_read_text(
    path,
    maximum_bytes=25 * 1024 * 1024,
):
    path = Path(
        path
    )

    if path.stat().st_size > maximum_bytes:
        return None

    try:
        return path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

    except Exception:
        return None


def file_sha256(
    path,
    block_size=8 * 1024 * 1024,
):
    path = Path(
        path
    )

    hasher = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            block = file.read(
                block_size
            )

            if not block:
                break

            hasher.update(
                block
            )

    return hasher.hexdigest()


def safe_package_version(
    package_name,
):
    try:
        return version(
            package_name
        )

    except PackageNotFoundError:
        return None


for abandoned_file in OUTPUT_ROOT.rglob(
    "*.temporary"
):
    try:
        abandoned_file.unlink()

    except Exception:
        pass


# ============================================================
# 6. General normalization helpers
# ============================================================

NUM_CLASSES = 4

CLASS_LABELS = [
    0,
    1,
    2,
    3,
]


def normalize_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(
            value
        ).strip().lower(),
    ).strip(
        "_"
    )


def find_column(
    dataframe,
    exact_aliases,
    contains_aliases=None,
):
    normalized_to_original = {
        normalize_name(
            column
        ): column

        for column in (
            dataframe.columns
        )
    }

    for alias in exact_aliases:
        normalized_alias = normalize_name(
            alias
        )

        if normalized_alias in normalized_to_original:
            return normalized_to_original[
                normalized_alias
            ]

    if contains_aliases is None:
        contains_aliases = []

    for normalized_column, original_column in (
        normalized_to_original.items()
    ):
        if any(
            normalize_name(
                alias
            )
            in
            normalized_column

            for alias in contains_aliases
        ):
            return original_column

    return None


def identify_probability_columns(
    dataframe,
):
    probability_map = {}

    for column in dataframe.columns:
        normalized = normalize_name(
            column
        )

        patterns = [
            r"^prob_?([0-3])$",
            r"^probability_?([0-3])$",
            r"^p_?([0-3])$",
            r"^class_?([0-3])_?prob$",
            r"^class_?([0-3])_?probability$",
            r"^softmax_?([0-3])$",
        ]

        for pattern in patterns:
            match = re.match(
                pattern,
                normalized,
            )

            if match:
                class_index = int(
                    match.group(
                        1
                    )
                )

                probability_map[
                    class_index
                ] = column

                break

    if set(
        probability_map.keys()
    ) == set(
        CLASS_LABELS
    ):
        return [
            probability_map[
                class_index
            ]

            for class_index in (
                CLASS_LABELS
            )
        ]

    return []


def parse_patient_id_from_path(
    path_value,
):
    if path_value is None:
        return None

    match = re.search(
        r"UC_patient_(\d+)_\d+",
        str(
            path_value
        ),
        flags=re.IGNORECASE,
    )

    if match:
        return int(
            match.group(
                1
            )
        )

    return None


def safe_mode_low(
    values,
):
    series = pd.Series(
        values
    ).dropna()

    if series.empty:
        return np.nan

    numeric_values = pd.to_numeric(
        series,
        errors="coerce",
    ).dropna()

    if numeric_values.empty:
        return np.nan

    modes = numeric_values.mode()

    if modes.empty:
        return int(
            numeric_values.iloc[
                0
            ]
        )

    return int(
        modes.min()
    )


def normalize_prediction_dataframe(
    dataframe,
    expected_level,
):
    dataframe = dataframe.copy()

    unnamed_columns = [
        column

        for column in dataframe.columns

        if normalize_name(
            column
        ).startswith(
            "unnamed"
        )
    ]

    if unnamed_columns:
        dataframe = dataframe.drop(
            columns=unnamed_columns
        )

    path_column = find_column(
        dataframe,

        exact_aliases=[
            "path",
            "image_path",
            "filepath",
            "file_path",
            "filename",
        ],
    )

    patient_id_column = find_column(
        dataframe,

        exact_aliases=[
            "patient_id",
            "patient",
            "pid",
            "group",
            "group_id",
        ],

        contains_aliases=[
            "patient_id",
        ],
    )

    if expected_level == "patient":
        truth_column = find_column(
            dataframe,

            exact_aliases=[
                "y_true",
                "patient_y_true",
                "patient_label",
                "true_label",
                "label",
                "target",
            ],
        )

    else:
        truth_column = find_column(
            dataframe,

            exact_aliases=[
                "y_true",
                "label",
                "target",
                "true_label",
                "image_y_true",
            ],
        )

    prediction_column = find_column(
        dataframe,

        exact_aliases=[
            "y_pred",
            "prediction",
            "pred",
            "predicted_label",
            "predicted_class",
            "class_prediction",
        ],
    )

    fold_column = find_column(
        dataframe,

        exact_aliases=[
            "fold",
            "cv_fold",
            "fold_id",
        ],
    )

    probability_columns = (
        identify_probability_columns(
            dataframe
        )
    )

    output = pd.DataFrame(
        index=dataframe.index
    )

    if path_column is not None:
        output[
            "path"
        ] = dataframe[
            path_column
        ].astype(
            str
        )

    if patient_id_column is not None:
        output[
            "patient_id"
        ] = pd.to_numeric(
            dataframe[
                patient_id_column
            ],
            errors="coerce",
        )

    elif path_column is not None:
        output[
            "patient_id"
        ] = dataframe[
            path_column
        ].map(
            parse_patient_id_from_path
        )

    if truth_column is not None:
        output[
            "y_true"
        ] = pd.to_numeric(
            dataframe[
                truth_column
            ],
            errors="coerce",
        )

    if prediction_column is not None:
        output[
            "y_pred"
        ] = pd.to_numeric(
            dataframe[
                prediction_column
            ],
            errors="coerce",
        )

    if fold_column is not None:
        output[
            "fold"
        ] = pd.to_numeric(
            dataframe[
                fold_column
            ],
            errors="coerce",
        )

    if probability_columns:
        for class_index, column in enumerate(
            probability_columns
        ):
            output[
                f"prob_{class_index}"
            ] = pd.to_numeric(
                dataframe[
                    column
                ],
                errors="coerce",
            )

        probability_matrix = output[
            [
                "prob_0",
                "prob_1",
                "prob_2",
                "prob_3",
            ]
        ].to_numpy(
            dtype=float
        )

        if (
            "y_pred"
            not in
            output.columns
        ):
            output[
                "y_pred"
            ] = np.argmax(
                probability_matrix,
                axis=1,
            )

    required_columns = [
        "y_true",
        "y_pred",
    ]

    missing_required = [
        column

        for column in required_columns

        if column not in output.columns
    ]

    if missing_required:
        raise ValueError(
            "Could not normalize prediction dataframe. "
            f"Missing: {missing_required}. "
            f"Available columns: {list(dataframe.columns)}"
        )

    output = output.dropna(
        subset=[
            "y_true",
            "y_pred",
        ]
    ).copy()

    output[
        "y_true"
    ] = output[
        "y_true"
    ].astype(
        int
    )

    output[
        "y_pred"
    ] = output[
        "y_pred"
    ].astype(
        int
    )

    if "patient_id" in output.columns:
        output = output.dropna(
            subset=[
                "patient_id",
            ]
        ).copy()

        output[
            "patient_id"
        ] = output[
            "patient_id"
        ].astype(
            int
        )

    if "fold" in output.columns:
        output[
            "fold"
        ] = output[
            "fold"
        ].astype(
            int
        )

    return output.reset_index(
        drop=True
    )


# ============================================================
# 7. Metric helpers
# ============================================================

def compute_metrics(
    y_true,
    y_pred,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    if len(
        y_true
    ) == 0:
        raise ValueError(
            "Cannot calculate metrics on zero rows."
        )

    qwk = cohen_kappa_score(
        y_true,
        y_pred,
        labels=CLASS_LABELS,
        weights="quadratic",
    )

    return {
        "qwk": (
            float(
                qwk
            )
            if np.isfinite(
                qwk
            )
            else np.nan
        ),

        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),

        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=CLASS_LABELS,
                average="macro",
                zero_division=0,
            )
        ),

        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "n_samples": int(
            len(
                y_true
            )
        ),
    }


def derive_patient_predictions_from_images(
    image_dataframe,
):
    if (
        "patient_id"
        not in
        image_dataframe.columns
    ):
        raise ValueError(
            "patient_id is required to derive "
            "patient predictions."
        )

    probability_columns = [
        f"prob_{class_index}"

        for class_index in (
            CLASS_LABELS
        )
    ]

    has_probabilities = all(
        column in image_dataframe.columns

        for column in probability_columns
    )

    rows = []

    for patient_id, patient_images in (
        image_dataframe
        .groupby(
            "patient_id",
            sort=True,
        )
    ):
        patient_truth = safe_mode_low(
            patient_images[
                "y_true"
            ]
        )

        if has_probabilities:
            mean_probabilities = (
                patient_images[
                    probability_columns
                ]
                .mean(
                    axis=0
                )
                .to_numpy(
                    dtype=float
                )
            )

            patient_prediction = int(
                np.argmax(
                    mean_probabilities
                )
            )

            row = {
                "patient_id": int(
                    patient_id
                ),

                "y_true": int(
                    patient_truth
                ),

                "y_pred": (
                    patient_prediction
                ),

                "n_images": int(
                    len(
                        patient_images
                    )
                ),
            }

            for class_index in CLASS_LABELS:
                row[
                    f"prob_{class_index}"
                ] = float(
                    mean_probabilities[
                        class_index
                    ]
                )

        else:
            patient_prediction = safe_mode_low(
                patient_images[
                    "y_pred"
                ]
            )

            row = {
                "patient_id": int(
                    patient_id
                ),

                "y_true": int(
                    patient_truth
                ),

                "y_pred": int(
                    patient_prediction
                ),

                "n_images": int(
                    len(
                        patient_images
                    )
                ),
            }

        rows.append(
            row
        )

    return pd.DataFrame(
        rows
    ).sort_values(
        "patient_id"
    ).reset_index(
        drop=True
    )


# ============================================================
# 8. Artifact inventory
# ============================================================

INVENTORY_EXTENSIONS = {
    ".csv",
    ".json",
    ".txt",
    ".md",
    ".py",
    ".ipynb",
    ".zip",
    ".pt",
    ".pth",
    ".png",
    ".pdf",
}


inventory_rows = []


for path in LEGACY_ROOT.rglob(
    "*"
):
    if not path.is_file():
        continue

    if (
        ".ipynb_checkpoints"
        in path.parts
    ):
        continue

    if (
        path.suffix.lower()
        not in
        INVENTORY_EXTENSIONS
    ):
        continue

    relative_path = path.relative_to(
        LEGACY_ROOT
    )

    inventory_rows.append({
        "file_name": (
            path.name
        ),

        "file_type": (
            path.suffix.lower()
        ),

        "relative_path": str(
            relative_path
        ),

        "full_path": str(
            path
        ),

        "size_bytes": int(
            path.stat().st_size
        ),

        "modified_time": datetime.fromtimestamp(
            path.stat().st_mtime,
            tz=timezone.utc,
        ).isoformat(),

        "checkpoint_content_loaded": False,
    })


artifact_inventory_df = (
    pd.DataFrame(
        inventory_rows
    )
    .sort_values(
        [
            "file_type",
            "relative_path",
        ]
    )
    .reset_index(
        drop=True
    )
)


atomic_csv_save(
    artifact_inventory_df,

    OUTPUT_ROOT
    / "legacy_artifact_inventory.csv",

    index=False,
)


print(
    "\nLegacy files inventoried:",
    len(
        artifact_inventory_df
    ),
)


# ============================================================
# 9. Prediction-file discovery
# ============================================================

EXACT_FILE_NAMES = {
    "oof_image": [
        "oof_predictions.csv",
        "oof_image_predictions.csv",
    ],

    "oof_patient": [
        "oof_patient_predictions.csv",
        "patient_oof_predictions.csv",
    ],

    "test_image": [
        "ensemble_test_predictions.csv",
        "test_image_predictions.csv",
        "test_predictions.csv",
    ],

    "test_patient": [
        "ensemble_test_patient_predictions.csv",
        "test_patient_predictions.csv",
    ],
}


def prediction_file_score(
    path,
    kind,
):
    name = path.name.lower()

    score = 0

    exact_names = EXACT_FILE_NAMES[
        kind
    ]

    if name in exact_names:
        score += (
            1000
            -
            exact_names.index(
                name
            )
        )

    if "prediction" in name:
        score += 50

    if "probabilit" in name:
        score += 10

    if kind.startswith(
        "oof"
    ) and "oof" in name:
        score += 100

    if kind.startswith(
        "test"
    ) and "test" in name:
        score += 100

    if kind.endswith(
        "patient"
    ) and "patient" in name:
        score += 150

    if kind.endswith(
        "image"
    ) and "patient" not in name:
        score += 40

    if (
        kind.endswith(
            "image"
        )
        and
        "patient"
        in name
    ):
        score -= 300

    if (
        kind.endswith(
            "patient"
        )
        and
        "patient"
        not in name
    ):
        score -= 100

    if "fold" in name:
        score -= 250

    if "history" in name:
        score -= 1000

    if "summary" in name:
        score -= 500

    if "confusion" in name:
        score -= 1000

    if "all_test_fold" in name:
        score -= 150

    if (
        "aggregation_sensitivity"
        in
        str(
            path
        )
    ):
        score -= 500

    return score


def discover_prediction_file(
    experiment_directory,
    kind,
):
    experiment_directory = Path(
        experiment_directory
    )

    if not experiment_directory.exists():
        return None, []

    candidates = []

    for path in experiment_directory.rglob(
        "*.csv"
    ):
        score = prediction_file_score(
            path,
            kind,
        )

        if score > 0:
            candidates.append(
                (
                    score,
                    path,
                )
            )

    candidates = sorted(
        candidates,
        key=lambda item: (
            -item[
                0
            ],
            str(
                item[
                    1
                ]
            ),
        ),
    )

    if not candidates:
        return None, []

    return (
        candidates[
            0
        ][
            1
        ],

        candidates,
    )


prediction_file_catalog_rows = []

selected_prediction_files = {}


for experiment_key, experiment_info in (
    LEGACY_EXPERIMENTS.items()
):
    experiment_directory = experiment_info[
        "path"
    ]

    selected_prediction_files[
        experiment_key
    ] = {}

    for kind in [
        "oof_image",
        "oof_patient",
        "test_image",
        "test_patient",
    ]:
        selected_file, candidates = (
            discover_prediction_file(
                experiment_directory,
                kind,
            )
        )

        selected_prediction_files[
            experiment_key
        ][
            kind
        ] = selected_file

        if selected_file is None:
            prediction_file_catalog_rows.append({
                "experiment_key": (
                    experiment_key
                ),

                "kind": (
                    kind
                ),

                "selected": False,

                "score": np.nan,

                "path": "",

                "status": (
                    "not_found"
                ),
            })

        else:
            for rank, (
                score,
                path,
            ) in enumerate(
                candidates[
                    :10
                ],
                start=1,
            ):
                prediction_file_catalog_rows.append({
                    "experiment_key": (
                        experiment_key
                    ),

                    "kind": (
                        kind
                    ),

                    "selected": bool(
                        path
                        ==
                        selected_file
                    ),

                    "candidate_rank": int(
                        rank
                    ),

                    "score": int(
                        score
                    ),

                    "path": str(
                        path
                    ),

                    "status": (
                        "selected"
                        if
                        path
                        ==
                        selected_file
                        else
                        "alternative"
                    ),
                })


prediction_file_catalog_df = pd.DataFrame(
    prediction_file_catalog_rows
)


atomic_csv_save(
    prediction_file_catalog_df,

    OUTPUT_ROOT
    / "legacy_prediction_file_catalog.csv",

    index=False,
)


# ============================================================
# 10. Configuration inventory
# ============================================================

def flatten_dictionary(
    value,
    prefix="",
):
    flattened = {}

    if isinstance(
        value,
        dict,
    ):
        for key, child_value in (
            value.items()
        ):
            child_prefix = (
                f"{prefix}.{key}"
                if prefix
                else str(
                    key
                )
            )

            flattened.update(
                flatten_dictionary(
                    child_value,
                    prefix=child_prefix,
                )
            )

    elif isinstance(
        value,
        list,
    ):
        flattened[
            prefix
        ] = json.dumps(
            value,
            default=str,
        )

    else:
        flattened[
            prefix
        ] = value

    return flattened


config_inventory_rows = []

config_lookup = defaultdict(
    dict
)

selected_config_files = {}


for experiment_key, experiment_info in (
    LEGACY_EXPERIMENTS.items()
):
    experiment_directory = experiment_info[
        "path"
    ]

    config_candidates = []

    if experiment_directory.exists():
        config_candidates = sorted(
            experiment_directory.rglob(
                "*.json"
            )
        )

    preferred_config = None

    for candidate_name in [
        "config_used.json",
        "config.json",
        "experiment_config.json",
    ]:
        matching = [
            path

            for path in config_candidates

            if path.name
            ==
            candidate_name
        ]

        if matching:
            preferred_config = matching[
                0
            ]

            break

    selected_config_files[
        experiment_key
    ] = preferred_config

    if preferred_config is None:
        continue

    try:
        config_payload = safe_read_json(
            preferred_config
        )

        flattened = flatten_dictionary(
            config_payload
        )

        for parameter_path, parameter_value in (
            flattened.items()
        ):
            config_inventory_rows.append({
                "experiment_key": (
                    experiment_key
                ),

                "config_file": str(
                    preferred_config
                ),

                "parameter_path": (
                    parameter_path
                ),

                "parameter_leaf": (
                    parameter_path.split(
                        "."
                    )[
                        -1
                    ]
                ),

                "value": (
                    parameter_value
                ),
            })

            config_lookup[
                experiment_key
            ][
                parameter_path
            ] = (
                parameter_value
            )

    except Exception as error:
        config_inventory_rows.append({
            "experiment_key": (
                experiment_key
            ),

            "config_file": str(
                preferred_config
            ),

            "parameter_path": "",

            "parameter_leaf": "",

            "value": "",

            "error": repr(
                error
            ),
        })


config_inventory_df = pd.DataFrame(
    config_inventory_rows
)


atomic_csv_save(
    config_inventory_df,

    OUTPUT_ROOT
    / "legacy_config_inventory.csv",

    index=False,
)


# ============================================================
# 11. Recompute main legacy metrics
# ============================================================

metrics_rows = []

patient_consistency_rows = []

prediction_cache = {}

metric_lookup = {}


def append_metric_rows(
    experiment_key,
    split,
    level,
    source_type,
    source_path,
    dataframe,
):
    metrics = compute_metrics(
        dataframe[
            "y_true"
        ],
        dataframe[
            "y_pred"
        ],
    )

    for metric_name, metric_value in (
        metrics.items()
    ):
        metrics_rows.append({
            "experiment_key": (
                experiment_key
            ),

            "split": (
                split
            ),

            "level": (
                level
            ),

            "metric": (
                metric_name
            ),

            "value": (
                metric_value
            ),

            "source_type": (
                source_type
            ),

            "source_path": str(
                source_path
            ),
        })

        metric_lookup[
            (
                experiment_key,
                split,
                level,
                metric_name,
            )
        ] = {
            "value": (
                metric_value
            ),

            "source_type": (
                source_type
            ),

            "source_path": str(
                source_path
            ),
        }


for experiment_key, experiment_info in (
    LEGACY_EXPERIMENTS.items()
):
    for split in [
        "oof",
        "test",
    ]:
        image_kind = (
            "oof_image"
            if split
            ==
            "oof"
            else
            "test_image"
        )

        patient_kind = (
            "oof_patient"
            if split
            ==
            "oof"
            else
            "test_patient"
        )

        image_file = (
            selected_prediction_files[
                experiment_key
            ][
                image_kind
            ]
        )

        patient_file = (
            selected_prediction_files[
                experiment_key
            ][
                patient_kind
            ]
        )

        normalized_image_df = None
        explicit_patient_df = None
        derived_patient_df = None

        if image_file is not None:
            try:
                raw_image_df = safe_read_csv(
                    image_file
                )

                normalized_image_df = (
                    normalize_prediction_dataframe(
                        raw_image_df,
                        expected_level="image",
                    )
                )

                append_metric_rows(
                    experiment_key=experiment_key,
                    split=split,
                    level="image",
                    source_type=(
                        "explicit_image_prediction_file"
                    ),
                    source_path=image_file,
                    dataframe=normalized_image_df,
                )

                if (
                    "patient_id"
                    in
                    normalized_image_df.columns
                ):
                    derived_patient_df = (
                        derive_patient_predictions_from_images(
                            normalized_image_df
                        )
                    )

                    append_metric_rows(
                        experiment_key=experiment_key,
                        split=split,
                        level="patient_derived",
                        source_type=(
                            "derived_from_image_predictions"
                        ),
                        source_path=image_file,
                        dataframe=derived_patient_df,
                    )

            except Exception as error:
                metrics_rows.append({
                    "experiment_key": (
                        experiment_key
                    ),

                    "split": (
                        split
                    ),

                    "level": (
                        "image"
                    ),

                    "metric": (
                        "read_or_normalization_error"
                    ),

                    "value": np.nan,

                    "source_type": (
                        "error"
                    ),

                    "source_path": str(
                        image_file
                    ),

                    "error": repr(
                        error
                    ),
                })

        if patient_file is not None:
            try:
                raw_patient_df = safe_read_csv(
                    patient_file
                )

                explicit_patient_df = (
                    normalize_prediction_dataframe(
                        raw_patient_df,
                        expected_level="patient",
                    )
                )

                append_metric_rows(
                    experiment_key=experiment_key,
                    split=split,
                    level="patient",
                    source_type=(
                        "explicit_patient_prediction_file"
                    ),
                    source_path=patient_file,
                    dataframe=explicit_patient_df,
                )

            except Exception as error:
                metrics_rows.append({
                    "experiment_key": (
                        experiment_key
                    ),

                    "split": (
                        split
                    ),

                    "level": (
                        "patient"
                    ),

                    "metric": (
                        "read_or_normalization_error"
                    ),

                    "value": np.nan,

                    "source_type": (
                        "error"
                    ),

                    "source_path": str(
                        patient_file
                    ),

                    "error": repr(
                        error
                    ),
                })

        elif derived_patient_df is not None:
            # Promote derived patient results to the main
            # patient lookup only when no explicit patient
            # file exists.
            derived_metrics = compute_metrics(
                derived_patient_df[
                    "y_true"
                ],
                derived_patient_df[
                    "y_pred"
                ],
            )

            for metric_name, metric_value in (
                derived_metrics.items()
            ):
                metric_lookup[
                    (
                        experiment_key,
                        split,
                        "patient",
                        metric_name,
                    )
                ] = {
                    "value": (
                        metric_value
                    ),

                    "source_type": (
                        "derived_from_image_predictions"
                    ),

                    "source_path": str(
                        image_file
                    ),
                }

        if (
            explicit_patient_df is not None
            and
            derived_patient_df is not None
            and
            "patient_id"
            in
            explicit_patient_df.columns
        ):
            explicit_aligned = (
                explicit_patient_df[
                    [
                        "patient_id",
                        "y_true",
                        "y_pred",
                    ]
                ]
                .rename(
                    columns={
                        "y_true": (
                            "explicit_y_true"
                        ),

                        "y_pred": (
                            "explicit_y_pred"
                        ),
                    }
                )
            )

            derived_aligned = (
                derived_patient_df[
                    [
                        "patient_id",
                        "y_true",
                        "y_pred",
                    ]
                ]
                .rename(
                    columns={
                        "y_true": (
                            "derived_y_true"
                        ),

                        "y_pred": (
                            "derived_y_pred"
                        ),
                    }
                )
            )

            alignment = explicit_aligned.merge(
                derived_aligned,
                on="patient_id",
                how="inner",
                validate="one_to_one",
            )

            patient_consistency_rows.append({
                "experiment_key": (
                    experiment_key
                ),

                "split": (
                    split
                ),

                "n_explicit_patients": int(
                    len(
                        explicit_aligned
                    )
                ),

                "n_derived_patients": int(
                    len(
                        derived_aligned
                    )
                ),

                "n_aligned_patients": int(
                    len(
                        alignment
                    )
                ),

                "n_truth_disagreements": int(
                    (
                        alignment[
                            "explicit_y_true"
                        ]
                        !=
                        alignment[
                            "derived_y_true"
                        ]
                    ).sum()
                ),

                "n_prediction_disagreements": int(
                    (
                        alignment[
                            "explicit_y_pred"
                        ]
                        !=
                        alignment[
                            "derived_y_pred"
                        ]
                    ).sum()
                ),

                "explicit_patient_file": str(
                    patient_file
                ),

                "image_source_file": str(
                    image_file
                ),
            })

        if experiment_key == "A3_no_class_weights":
            if normalized_image_df is not None:
                prediction_cache[
                    (
                        experiment_key,
                        split,
                        "image",
                    )
                ] = normalized_image_df

            if explicit_patient_df is not None:
                prediction_cache[
                    (
                        experiment_key,
                        split,
                        "patient",
                    )
                ] = explicit_patient_df

            elif derived_patient_df is not None:
                prediction_cache[
                    (
                        experiment_key,
                        split,
                        "patient",
                    )
                ] = derived_patient_df


metrics_recomputed_df = pd.DataFrame(
    metrics_rows
)


patient_consistency_df = pd.DataFrame(
    patient_consistency_rows
)


atomic_csv_save(
    metrics_recomputed_df,

    OUTPUT_ROOT
    / "legacy_metrics_recomputed.csv",

    index=False,
)


atomic_csv_save(
    patient_consistency_df,

    OUTPUT_ROOT
    / "legacy_patient_prediction_consistency.csv",

    index=False,
)


# ============================================================
# 12. Recover fold-level patient counts
# ============================================================

fold_count_rows = []

fold_count_lookup = {}


A3_DIRECTORY = (
    LEGACY_EXPERIMENTS[
        "A3_no_class_weights"
    ][
        "path"
    ]
)


for fold in range(
    5
):
    candidate_files = sorted(
        A3_DIRECTORY.rglob(
            f"val_predictions_fold{fold}.csv"
        )
    )

    if not candidate_files:
        candidate_files = sorted(
            A3_DIRECTORY.rglob(
                f"*val*fold*{fold}*.csv"
            )
        )

    if not candidate_files:
        continue

    selected_file = candidate_files[
        0
    ]

    try:
        raw_df = safe_read_csv(
            selected_file
        )

        normalized_df = (
            normalize_prediction_dataframe(
                raw_df,
                expected_level="image",
            )
        )

        if (
            "patient_id"
            in
            normalized_df.columns
        ):
            patient_count = int(
                normalized_df[
                    "patient_id"
                ].nunique()
            )

            fold_count_rows.append({
                "fold": int(
                    fold
                ),

                "n_patients": (
                    patient_count
                ),

                "source_path": str(
                    selected_file
                ),
            })

            fold_count_lookup[
                fold
            ] = {
                "value": (
                    patient_count
                ),

                "source_path": str(
                    selected_file
                ),
            }

    except Exception as error:
        fold_count_rows.append({
            "fold": int(
                fold
            ),

            "n_patients": np.nan,

            "source_path": str(
                selected_file
            ),

            "error": repr(
                error
            ),
        })


fold_count_df = pd.DataFrame(
    fold_count_rows
)


atomic_csv_save(
    fold_count_df,

    OUTPUT_ROOT
    / "legacy_fold_patient_counts.csv",

    index=False,
)


# ============================================================
# 13. Recompute final A3 confusion and per-class metrics
# ============================================================

confusion_rows = []

class_metric_rows = []

confusion_lookup = {}

class_metric_lookup = {}


for split in [
    "oof",
    "test",
]:
    for level in [
        "image",
        "patient",
    ]:
        cache_key = (
            "A3_no_class_weights",
            split,
            level,
        )

        if cache_key not in prediction_cache:
            continue

        dataframe = prediction_cache[
            cache_key
        ]

        matrix = confusion_matrix(
            dataframe[
                "y_true"
            ],
            dataframe[
                "y_pred"
            ],
            labels=CLASS_LABELS,
        )

        precision, recall, f1_values, support = (
            precision_recall_fscore_support(
                dataframe[
                    "y_true"
                ],
                dataframe[
                    "y_pred"
                ],
                labels=CLASS_LABELS,
                zero_division=0,
            )
        )

        for true_class in CLASS_LABELS:
            for predicted_class in CLASS_LABELS:
                cell_value = int(
                    matrix[
                        true_class,
                        predicted_class,
                    ]
                )

                confusion_rows.append({
                    "experiment_key": (
                        "A3_no_class_weights"
                    ),

                    "split": (
                        split
                    ),

                    "level": (
                        level
                    ),

                    "true_class": int(
                        true_class
                    ),

                    "predicted_class": int(
                        predicted_class
                    ),

                    "count": (
                        cell_value
                    ),
                })

                confusion_lookup[
                    (
                        "A3_no_class_weights",
                        split,
                        level,
                        true_class,
                        predicted_class,
                    )
                ] = {
                    "value": (
                        cell_value
                    ),

                    "source_path": (
                        metric_lookup.get(
                            (
                                "A3_no_class_weights",
                                split,
                                level,
                                "qwk",
                            ),
                            {},
                        ).get(
                            "source_path",
                            "",
                        )
                    ),
                }

        for class_index in CLASS_LABELS:
            metric_values = {
                "precision": float(
                    precision[
                        class_index
                    ]
                ),

                "recall": float(
                    recall[
                        class_index
                    ]
                ),

                "f1": float(
                    f1_values[
                        class_index
                    ]
                ),

                "support": int(
                    support[
                        class_index
                    ]
                ),
            }

            for metric_name, metric_value in (
                metric_values.items()
            ):
                class_metric_rows.append({
                    "experiment_key": (
                        "A3_no_class_weights"
                    ),

                    "split": (
                        split
                    ),

                    "level": (
                        level
                    ),

                    "class_index": int(
                        class_index
                    ),

                    "metric": (
                        metric_name
                    ),

                    "value": (
                        metric_value
                    ),
                })

                class_metric_lookup[
                    (
                        "A3_no_class_weights",
                        split,
                        level,
                        class_index,
                        metric_name,
                    )
                ] = {
                    "value": (
                        metric_value
                    ),

                    "source_path": (
                        metric_lookup.get(
                            (
                                "A3_no_class_weights",
                                split,
                                level,
                                "qwk",
                            ),
                            {},
                        ).get(
                            "source_path",
                            "",
                        )
                    ),
                }


confusion_metrics_df = pd.DataFrame(
    confusion_rows
)


class_metrics_df = pd.DataFrame(
    class_metric_rows
)


atomic_csv_save(
    confusion_metrics_df,

    OUTPUT_ROOT
    / "legacy_confusion_metrics_recomputed.csv",

    index=False,
)


atomic_csv_save(
    class_metrics_df,

    OUTPUT_ROOT
    / "legacy_per_class_metrics_recomputed.csv",

    index=False,
)


# ============================================================
# 14. Search legacy code for aggregation definitions
# ============================================================

CODE_SEARCH_TERMS = [
    "trimmed",
    "trim_mean",
    "proportiontocut",
    "confidence-weighted",
    "confidence_weighted",
    "max-severity",
    "max_severity",
    "median probability",
    "median_probability",
    "majority vote",
    "majority_vote",
]


code_evidence_rows = []


source_search_roots = [
    CODE_ROOT,
    AGGREGATION_ROOT,
    A3_DIRECTORY,
]


source_extensions = {
    ".py",
    ".ipynb",
    ".json",
    ".txt",
    ".md",
}


for source_root in source_search_roots:
    if not source_root.exists():
        continue

    for path in source_root.rglob(
        "*"
    ):
        if not path.is_file():
            continue

        if (
            path.suffix.lower()
            not in
            source_extensions
        ):
            continue

        text = safe_read_text(
            path
        )

        if text is None:
            continue

        lower_text = text.lower()

        for term in CODE_SEARCH_TERMS:
            search_term = term.lower()

            start = 0

            occurrence_count = 0

            while True:
                index = lower_text.find(
                    search_term,
                    start,
                )

                if index < 0:
                    break

                snippet_start = max(
                    0,
                    index - 250,
                )

                snippet_end = min(
                    len(
                        text
                    ),
                    index + 500,
                )

                snippet = (
                    text[
                        snippet_start:snippet_end
                    ]
                    .replace(
                        "\n",
                        " "
                    )
                    .replace(
                        "\r",
                        " "
                    )
                )

                code_evidence_rows.append({
                    "source_path": str(
                        path
                    ),

                    "search_term": (
                        term
                    ),

                    "occurrence_index": int(
                        occurrence_count
                    ),

                    "snippet": (
                        snippet
                    ),
                })

                occurrence_count += 1

                start = (
                    index
                    +
                    len(
                        search_term
                    )
                )

                if occurrence_count >= 20:
                    break


code_evidence_df = pd.DataFrame(
    code_evidence_rows
)


atomic_csv_save(
    code_evidence_df,

    OUTPUT_ROOT
    / "legacy_code_evidence.csv",

    index=False,
)


def infer_trim_fraction_from_code(
    evidence_dataframe,
):
    if evidence_dataframe.empty:
        return None

    combined_text = " ".join(
        evidence_dataframe[
            "snippet"
        ].astype(
            str
        ).tolist()
    )

    patterns = [
        r"proportiontocut\s*=\s*(0?\.\d+)",
        r"trim_fraction\s*=\s*(0?\.\d+)",
        r"trim_ratio\s*=\s*(0?\.\d+)",
        r"trim_proportion\s*=\s*(0?\.\d+)",
        r"trim_percent\s*=\s*(0?\.\d+)",
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            combined_text,
            flags=re.IGNORECASE,
        )

        if match:
            value = float(
                match.group(
                    1
                )
            )

            if (
                value >= 0
                and value < 0.5
            ):
                return value

    return None


INFERRED_TRIM_FRACTION = (
    infer_trim_fraction_from_code(
        code_evidence_df
    )
)


print(
    "\nInferred legacy trim fraction:",
    INFERRED_TRIM_FRACTION,
)


# ============================================================
# 15. Parse saved aggregation summary files
# ============================================================

def canonical_aggregation_rule(
    value,
):
    normalized = normalize_name(
        value
    )

    if (
        "trim"
        in
        normalized
    ):
        return (
            "trimmed_mean_probability"
        )

    if (
        "confidence"
        in
        normalized
    ):
        return (
            "confidence_weighted_mean"
        )

    if (
        "majority"
        in
        normalized
        or
        "vote"
        in
        normalized
    ):
        return (
            "majority_vote"
        )

    if (
        "max"
        in
        normalized
        and
        (
            "severity"
            in
            normalized
            or
            "pred"
            in
            normalized
        )
    ):
        return (
            "max_severity_prediction"
        )

    if (
        "median"
        in
        normalized
    ):
        return (
            "median_probability"
        )

    if (
        "mean"
        in
        normalized
        or
        "average"
        in
        normalized
    ):
        return (
            "mean_probability"
        )

    return normalized


def canonical_split(
    value,
):
    normalized = normalize_name(
        value
    )

    if (
        "oof"
        in
        normalized
        or
        "validation"
        in
        normalized
        or
        "val"
        ==
        normalized
    ):
        return "oof"

    if (
        "test"
        in
        normalized
    ):
        return "test"

    return normalized


def parse_aggregation_summary_file(
    path,
):
    dataframe = safe_read_csv(
        path
    )

    normalized_columns = {
        normalize_name(
            column
        ): column

        for column in (
            dataframe.columns
        )
    }

    split_column = None
    rule_column = None

    for normalized, original in (
        normalized_columns.items()
    ):
        if split_column is None and (
            normalized
            in {
                "split",
                "dataset_split",
                "evaluation_split",
            }
            or
            normalized.endswith(
                "_split"
            )
        ):
            split_column = original

        if rule_column is None and (
            "aggregation"
            in
            normalized
            or
            normalized
            in {
                "rule",
                "method",
                "aggregation_rule",
            }
        ):
            rule_column = original

    metric_columns = {}

    for normalized, original in (
        normalized_columns.items()
    ):
        if (
            "qwk"
            in
            normalized
        ):
            metric_columns[
                "qwk"
            ] = original

        elif (
            "mae"
            in
            normalized
        ):
            metric_columns[
                "mae"
            ] = original

        elif (
            "macro"
            in
            normalized
            and
            "f1"
            in
            normalized
        ):
            metric_columns[
                "macro_f1"
            ] = original

        elif normalized in {
            "f1",
            "patient_f1",
            "patient_level_f1",
        }:
            metric_columns[
                "macro_f1"
            ] = original

        elif (
            "accuracy"
            in
            normalized
            or
            normalized.endswith(
                "_acc"
            )
            or
            normalized
            ==
            "acc"
        ):
            metric_columns[
                "accuracy"
            ] = original

    if (
        rule_column is None
        or
        len(
            metric_columns
        )
        < 2
    ):
        return None

    output_rows = []

    for _, row in (
        dataframe.iterrows()
    ):
        if pd.isna(
            row[
                rule_column
            ]
        ):
            continue

        split = (
            canonical_split(
                row[
                    split_column
                ]
            )
            if split_column is not None
            else
            (
                "test"
                if
                "test"
                in
                path.name.lower()
                else
                "oof"
                if
                "oof"
                in
                path.name.lower()
                else
                "unknown"
            )
        )

        aggregation_rule = (
            canonical_aggregation_rule(
                row[
                    rule_column
                ]
            )
        )

        for metric_name, metric_column in (
            metric_columns.items()
        ):
            metric_value = pd.to_numeric(
                pd.Series(
                    [
                        row[
                            metric_column
                        ]
                    ]
                ),
                errors="coerce",
            ).iloc[
                0
            ]

            if pd.isna(
                metric_value
            ):
                continue

            output_rows.append({
                "source_path": str(
                    path
                ),

                "split": (
                    split
                ),

                "aggregation_rule": (
                    aggregation_rule
                ),

                "metric": (
                    metric_name
                ),

                "value": float(
                    metric_value
                ),

                "source_type": (
                    "saved_legacy_aggregation_summary"
                ),
            })

    if not output_rows:
        return None

    return pd.DataFrame(
        output_rows
    )


aggregation_summary_candidates = []

aggregation_summary_parsed_parts = []


if AGGREGATION_ROOT.exists():
    for path in AGGREGATION_ROOT.rglob(
        "*.csv"
    ):
        try:
            parsed = (
                parse_aggregation_summary_file(
                    path
                )
            )

            if parsed is not None:
                aggregation_summary_candidates.append({
                    "source_path": str(
                        path
                    ),

                    "n_parsed_rows": int(
                        len(
                            parsed
                        )
                    ),

                    "n_rules": int(
                        parsed[
                            "aggregation_rule"
                        ].nunique()
                    ),

                    "n_splits": int(
                        parsed[
                            "split"
                        ].nunique()
                    ),
                })

                aggregation_summary_parsed_parts.append(
                    parsed
                )

        except Exception:
            pass


if aggregation_summary_parsed_parts:
    aggregation_saved_summary_df = pd.concat(
        aggregation_summary_parsed_parts,
        ignore_index=True,
    )

    aggregation_saved_summary_df = (
        aggregation_saved_summary_df
        .sort_values(
            [
                "split",
                "aggregation_rule",
                "metric",
                "source_path",
            ]
        )
        .drop_duplicates(
            subset=[
                "split",
                "aggregation_rule",
                "metric",
            ],
            keep="first",
        )
        .reset_index(
            drop=True
        )
    )

else:
    aggregation_saved_summary_df = pd.DataFrame(
        columns=[
            "source_path",
            "split",
            "aggregation_rule",
            "metric",
            "value",
            "source_type",
        ]
    )


aggregation_summary_candidates_df = pd.DataFrame(
    aggregation_summary_candidates
)


atomic_csv_save(
    aggregation_saved_summary_df,

    OUTPUT_ROOT
    / "legacy_aggregation_saved_summary.csv",

    index=False,
)


atomic_csv_save(
    aggregation_summary_candidates_df,

    OUTPUT_ROOT
    / "legacy_aggregation_summary_file_candidates.csv",

    index=False,
)


# ============================================================
# 16. Independently recompute available aggregation rules
# ============================================================

def aggregate_legacy_rule(
    image_dataframe,
    aggregation_rule,
    trim_fraction=None,
):
    required_columns = [
        "patient_id",
        "y_true",
        "y_pred",
    ]

    for column in required_columns:
        if column not in image_dataframe.columns:
            raise ValueError(
                f"Missing required column: {column}"
            )

    probability_columns = [
        f"prob_{class_index}"

        for class_index in CLASS_LABELS
    ]

    has_probabilities = all(
        column in image_dataframe.columns

        for column in probability_columns
    )

    rows = []

    for patient_id, patient_images in (
        image_dataframe
        .groupby(
            "patient_id",
            sort=True,
        )
    ):
        patient_truth = safe_mode_low(
            patient_images[
                "y_true"
            ]
        )

        hard_predictions = patient_images[
            "y_pred"
        ].to_numpy(
            dtype=int
        )

        if aggregation_rule == "majority_vote":
            patient_prediction = safe_mode_low(
                hard_predictions
            )

        elif (
            aggregation_rule
            ==
            "max_severity_prediction"
        ):
            patient_prediction = int(
                hard_predictions.max()
            )

        else:
            if not has_probabilities:
                raise ValueError(
                    "Probability columns are required for "
                    f"{aggregation_rule}."
                )

            probability_matrix = (
                patient_images[
                    probability_columns
                ]
                .to_numpy(
                    dtype=float
                )
            )

            if (
                aggregation_rule
                ==
                "mean_probability"
            ):
                aggregate_probabilities = (
                    probability_matrix.mean(
                        axis=0
                    )
                )

            elif (
                aggregation_rule
                ==
                "median_probability"
            ):
                aggregate_probabilities = (
                    np.median(
                        probability_matrix,
                        axis=0,
                    )
                )

            elif (
                aggregation_rule
                ==
                "trimmed_mean_probability"
            ):
                if trim_fraction is None:
                    raise ValueError(
                        "Trim fraction was not identified "
                        "from legacy source code."
                    )

                aggregate_probabilities = trim_mean(
                    probability_matrix,
                    proportiontocut=trim_fraction,
                    axis=0,
                )

            elif (
                aggregation_rule
                ==
                "confidence_weighted_mean"
            ):
                confidence_weights = (
                    probability_matrix.max(
                        axis=1
                    )
                )

                if np.isclose(
                    confidence_weights.sum(),
                    0.0,
                ):
                    aggregate_probabilities = (
                        probability_matrix.mean(
                            axis=0
                        )
                    )

                else:
                    aggregate_probabilities = (
                        np.average(
                            probability_matrix,
                            axis=0,
                            weights=confidence_weights,
                        )
                    )

            else:
                raise ValueError(
                    f"Unknown aggregation rule: "
                    f"{aggregation_rule}"
                )

            patient_prediction = int(
                np.argmax(
                    aggregate_probabilities
                )
            )

        rows.append({
            "patient_id": int(
                patient_id
            ),

            "y_true": int(
                patient_truth
            ),

            "y_pred": int(
                patient_prediction
            ),
        })

    return pd.DataFrame(
        rows
    )


aggregation_recomputed_rows = []


for split in [
    "oof",
    "test",
]:
    cache_key = (
        "A3_no_class_weights",
        split,
        "image",
    )

    if cache_key not in prediction_cache:
        continue

    image_dataframe = prediction_cache[
        cache_key
    ]

    rule_definitions = [
        (
            "mean_probability",
            None,
            "independent_recomputation",
        ),

        (
            "median_probability",
            None,
            "independent_recomputation",
        ),

        (
            "majority_vote",
            None,
            "independent_recomputation",
        ),

        (
            "max_severity_prediction",
            None,
            "independent_recomputation",
        ),

        (
            "trimmed_mean_probability",
            INFERRED_TRIM_FRACTION,
            (
                "independent_recomputation_using_"
                "trim_fraction_from_source"
                if
                INFERRED_TRIM_FRACTION is not None
                else
                "not_recomputed_trim_fraction_unknown"
            ),
        ),

        (
            "confidence_weighted_mean",
            None,
            (
                "independent_recomputation_assuming_"
                "max_probability_weights"
            ),
        ),
    ]

    for (
        aggregation_rule,
        trim_fraction,
        source_type,
    ) in rule_definitions:
        if (
            aggregation_rule
            ==
            "trimmed_mean_probability"
            and
            trim_fraction is None
        ):
            continue

        try:
            patient_dataframe = (
                aggregate_legacy_rule(
                    image_dataframe=image_dataframe,
                    aggregation_rule=(
                        aggregation_rule
                    ),
                    trim_fraction=(
                        trim_fraction
                    ),
                )
            )

            metrics = compute_metrics(
                patient_dataframe[
                    "y_true"
                ],
                patient_dataframe[
                    "y_pred"
                ],
            )

            for metric_name, metric_value in (
                metrics.items()
            ):
                aggregation_recomputed_rows.append({
                    "split": (
                        split
                    ),

                    "aggregation_rule": (
                        aggregation_rule
                    ),

                    "metric": (
                        metric_name
                    ),

                    "value": (
                        metric_value
                    ),

                    "source_type": (
                        source_type
                    ),

                    "trim_fraction": (
                        trim_fraction
                    ),

                    "image_prediction_source": (
                        metric_lookup.get(
                            (
                                "A3_no_class_weights",
                                split,
                                "image",
                                "qwk",
                            ),
                            {},
                        ).get(
                            "source_path",
                            "",
                        )
                    ),
                })

        except Exception as error:
            aggregation_recomputed_rows.append({
                "split": (
                    split
                ),

                "aggregation_rule": (
                    aggregation_rule
                ),

                "metric": (
                    "recomputation_error"
                ),

                "value": np.nan,

                "source_type": (
                    source_type
                ),

                "trim_fraction": (
                    trim_fraction
                ),

                "error": repr(
                    error
                ),
            })


aggregation_recomputed_df = pd.DataFrame(
    aggregation_recomputed_rows
)


atomic_csv_save(
    aggregation_recomputed_df,

    OUTPUT_ROOT
    / "legacy_aggregation_recomputed.csv",

    index=False,
)


# Build aggregation lookup.
# Prefer independent recomputation for rules whose definition
# is unambiguous. Otherwise use a saved legacy summary.
aggregation_lookup = {}


unambiguous_rules = {
    "mean_probability",
    "median_probability",
    "majority_vote",
    "max_severity_prediction",
}


if not aggregation_recomputed_df.empty:
    for _, row in aggregation_recomputed_df.iterrows():
        if (
            row[
                "metric"
            ]
            ==
            "recomputation_error"
        ):
            continue

        key = (
            row[
                "split"
            ],
            row[
                "aggregation_rule"
            ],
            row[
                "metric"
            ],
        )

        if (
            row[
                "aggregation_rule"
            ]
            in
            unambiguous_rules
        ):
            aggregation_lookup[
                key
            ] = {
                "value": float(
                    row[
                        "value"
                    ]
                ),

                "source_type": (
                    row[
                        "source_type"
                    ]
                ),

                "source_path": (
                    row.get(
                        "image_prediction_source",
                        "",
                    )
                ),
            }


if not aggregation_saved_summary_df.empty:
    for _, row in aggregation_saved_summary_df.iterrows():
        key = (
            row[
                "split"
            ],
            row[
                "aggregation_rule"
            ],
            row[
                "metric"
            ],
        )

        if key not in aggregation_lookup:
            aggregation_lookup[
                key
            ] = {
                "value": float(
                    row[
                        "value"
                    ]
                ),

                "source_type": (
                    row[
                        "source_type"
                    ]
                ),

                "source_path": (
                    row[
                        "source_path"
                    ]
                ),
            }


# ============================================================
# 17. Manuscript numerical claim registry
# ============================================================

claim_rows = []


def add_claim(
    claim_id,
    manuscript_location,
    claim_kind,
    manuscript_value,
    reported_decimals=4,
    experiment_key=None,
    split=None,
    level=None,
    metric=None,
    aggregation_rule=None,
    true_class=None,
    predicted_class=None,
    class_index=None,
    fold=None,
    config_parameter=None,
    note="",
):
    claim_rows.append({
        "claim_id": (
            claim_id
        ),

        "manuscript_location": (
            manuscript_location
        ),

        "claim_kind": (
            claim_kind
        ),

        "manuscript_value": (
            manuscript_value
        ),

        "reported_decimals": (
            reported_decimals
        ),

        "experiment_key": (
            experiment_key
        ),

        "split": (
            split
        ),

        "level": (
            level
        ),

        "metric": (
            metric
        ),

        "aggregation_rule": (
            aggregation_rule
        ),

        "true_class": (
            true_class
        ),

        "predicted_class": (
            predicted_class
        ),

        "class_index": (
            class_index
        ),

        "fold": (
            fold
        ),

        "config_parameter": (
            config_parameter
        ),

        "note": (
            note
        ),
    })


# ------------------------------------------------------------
# 17A. Abstract final-model claims
# ------------------------------------------------------------

ABSTRACT_FINAL_VALUES = {
    (
        "test",
        "image",
        "qwk",
    ): 0.8759,

    (
        "test",
        "image",
        "mae",
    ): 0.1969,

    (
        "test",
        "patient",
        "qwk",
    ): 0.8213,

    (
        "test",
        "patient",
        "mae",
    ): 0.1977,
}


for (
    split,
    level,
    metric,
), value in ABSTRACT_FINAL_VALUES.items():
    add_claim(
        claim_id=(
            f"ABSTRACT_A3_{split}_{level}_{metric}"
        ),

        manuscript_location=(
            "Abstract, page 1"
        ),

        claim_kind=(
            "metric"
        ),

        manuscript_value=(
            value
        ),

        experiment_key=(
            "A3_no_class_weights"
        ),

        split=(
            split
        ),

        level=(
            level
        ),

        metric=(
            metric
        ),

        note=(
            "Legacy final-model abstract result; "
            "superseded by Route A."
        ),
    )


# ------------------------------------------------------------
# 17B. Table 2
# ------------------------------------------------------------

TABLE2_VALUES = {
    "M1_vit_oldordmae": {
        (
            "oof",
            "image",
            "qwk",
        ): 0.8402,

        (
            "oof",
            "patient",
            "qwk",
        ): 0.8327,

        (
            "test",
            "image",
            "qwk",
        ): 0.8461,

        (
            "test",
            "patient",
            "qwk",
        ): 0.7218,

        (
            "test",
            "patient",
            "mae",
        ): None,
    },

    "M2_swin_oldordmae": {
        (
            "oof",
            "image",
            "qwk",
        ): 0.8648,

        (
            "oof",
            "patient",
            "qwk",
        ): 0.8290,

        (
            "test",
            "image",
            "qwk",
        ): 0.8657,

        (
            "test",
            "patient",
            "qwk",
        ): 0.7613,

        (
            "test",
            "patient",
            "mae",
        ): None,
    },

    "M3_swin_orddist_a2": {
        (
            "oof",
            "image",
            "qwk",
        ): 0.8714,

        (
            "oof",
            "patient",
            "qwk",
        ): 0.8225,

        (
            "test",
            "image",
            "qwk",
        ): 0.8647,

        (
            "test",
            "patient",
            "qwk",
        ): 0.7946,

        (
            "test",
            "patient",
            "mae",
        ): 0.2093,
    },

    "M4_convnext_orddist_a2": {
        (
            "oof",
            "image",
            "qwk",
        ): 0.8703,

        (
            "oof",
            "patient",
            "qwk",
        ): 0.8508,

        (
            "test",
            "image",
            "qwk",
        ): 0.8663,

        (
            "test",
            "patient",
            "qwk",
        ): 0.7819,

        (
            "test",
            "patient",
            "mae",
        ): 0.2209,
    },

    "M5_maxvit_orddist_a2": {
        (
            "oof",
            "image",
            "qwk",
        ): 0.8684,

        (
            "oof",
            "patient",
            "qwk",
        ): 0.8367,

        (
            "test",
            "image",
            "qwk",
        ): 0.8783,

        (
            "test",
            "patient",
            "qwk",
        ): 0.7689,

        (
            "test",
            "patient",
            "mae",
        ): 0.2326,
    },
}


for experiment_key, claims in (
    TABLE2_VALUES.items()
):
    for (
        split,
        level,
        metric,
    ), value in claims.items():
        add_claim(
            claim_id=(
                f"TABLE2_{experiment_key}_"
                f"{split}_{level}_{metric}"
            ),

            manuscript_location=(
                "Table 2, page 9"
            ),

            claim_kind=(
                "metric"
            ),

            manuscript_value=(
                value
            ),

            experiment_key=(
                experiment_key
            ),

            split=(
                split
            ),

            level=(
                level
            ),

            metric=(
                metric
            ),

            note=(
                "Manuscript omitted this value."
                if value is None
                else
                ""
            ),
        )


# ------------------------------------------------------------
# 17C. Table 3 legacy ablation
# ------------------------------------------------------------

TABLE3_VALUES = {
    "A0_default": {
        "qwk": (
            0.8647,
            0.7946,
        ),

        "mae": (
            0.2272,
            0.2093,
        ),

        "macro_f1": (
            0.7187,
            0.7231,
        ),

        "accuracy": (
            0.7794,
            0.8372,
        ),
    },

    "A1_no_tta": {
        "qwk": (
            0.8735,
            0.7874,
        ),

        "mae": (
            0.2082,
            0.2209,
        ),

        "macro_f1": (
            0.7482,
            0.7358,
        ),

        "accuracy": (
            0.7977,
            0.8256,
        ),
    },

    "A2_reduced_aug": {
        "qwk": (
            0.8644,
            0.7649,
        ),

        "mae": (
            0.2212,
            0.2442,
        ),

        "macro_f1": (
            0.7213,
            0.6956,
        ),

        "accuracy": (
            0.7877,
            0.8023,
        ),
    },

    "A3_no_class_weights": {
        "qwk": (
            0.8759,
            0.8213,
        ),

        "mae": (
            0.1969,
            0.1977,
        ),

        "macro_f1": (
            0.7477,
            0.6536,
        ),

        "accuracy": (
            0.8078,
            0.8256,
        ),
    },
}


for experiment_key, metric_values in (
    TABLE3_VALUES.items()
):
    for metric_name, (
        image_value,
        patient_value,
    ) in metric_values.items():
        add_claim(
            claim_id=(
                f"TABLE3_{experiment_key}_"
                f"test_image_{metric_name}"
            ),

            manuscript_location=(
                "Table 3, page 10"
            ),

            claim_kind=(
                "metric"
            ),

            manuscript_value=(
                image_value
            ),

            experiment_key=(
                experiment_key
            ),

            split=(
                "test"
            ),

            level=(
                "image"
            ),

            metric=(
                metric_name
            ),
        )

        add_claim(
            claim_id=(
                f"TABLE3_{experiment_key}_"
                f"test_patient_{metric_name}"
            ),

            manuscript_location=(
                "Table 3, page 10"
            ),

            claim_kind=(
                "metric"
            ),

            manuscript_value=(
                patient_value
            ),

            experiment_key=(
                experiment_key
            ),

            split=(
                "test"
            ),

            level=(
                "patient"
            ),

            metric=(
                metric_name
            ),
        )


# ------------------------------------------------------------
# 17D. Table 4 final A3 OOF/test results
# ------------------------------------------------------------

TABLE4_VALUES = {
    "oof": {
        "image": {
            "qwk": 0.8729,
            "mae": 0.2132,
            "macro_f1": 0.7284,
            "accuracy": 0.7901,
        },

        "patient": {
            "qwk": 0.7715,
            "mae": 0.2280,
            "macro_f1": 0.7215,
            "accuracy": 0.8159,
        },
    },

    "test": {
        "image": {
            "qwk": 0.8759,
            "mae": 0.1969,
            "macro_f1": 0.7477,
            "accuracy": 0.8078,
        },

        "patient": {
            "qwk": 0.8213,
            "mae": 0.1977,
            "macro_f1": 0.6536,
            "accuracy": 0.8256,
        },
    },
}


for split, level_values in (
    TABLE4_VALUES.items()
):
    for level, metric_values in (
        level_values.items()
    ):
        for metric_name, value in (
            metric_values.items()
        ):
            add_claim(
                claim_id=(
                    f"TABLE4_A3_{split}_"
                    f"{level}_{metric_name}"
                ),

                manuscript_location=(
                    "Table 4, page 10"
                ),

                claim_kind=(
                    "metric"
                ),

                manuscript_value=(
                    value
                ),

                experiment_key=(
                    "A3_no_class_weights"
                ),

                split=(
                    split
                ),

                level=(
                    level
                ),

                metric=(
                    metric_name
                ),
            )


# ------------------------------------------------------------
# 17E. Supplementary Table S1
# ------------------------------------------------------------

S1_VALUES = {
    "oof": {
        "median_probability": {
            "qwk": 0.8322,
            "mae": 0.2134,
            "macro_f1": 0.7073,
            "accuracy": 0.8075,
        },

        "trimmed_mean_probability": {
            "qwk": 0.8030,
            "mae": 0.2280,
            "macro_f1": 0.7126,
            "accuracy": 0.8033,
        },

        "majority_vote": {
            "qwk": 0.7955,
            "mae": 0.2238,
            "macro_f1": 0.7092,
            "accuracy": 0.8117,
        },

        "mean_probability": {
            "qwk": 0.7715,
            "mae": 0.2280,
            "macro_f1": 0.7215,
            "accuracy": 0.8159,
        },

        "confidence_weighted_mean": {
            "qwk": 0.7077,
            "mae": 0.2636,
            "macro_f1": 0.6775,
            "accuracy": 0.8013,
        },

        "max_severity_prediction": {
            "qwk": 0.2159,
            "mae": 1.2573,
            "macro_f1": 0.2550,
            "accuracy": 0.2573,
        },
    },

    "test": {
        "mean_probability": {
            "qwk": 0.8213,
            "mae": 0.1977,
            "macro_f1": 0.6536,
            "accuracy": 0.8256,
        },

        "trimmed_mean_probability": {
            "qwk": 0.8074,
            "mae": 0.2093,
            "macro_f1": 0.6107,
            "accuracy": 0.8140,
        },

        "median_probability": {
            "qwk": 0.8063,
            "mae": 0.2093,
            "macro_f1": 0.6734,
            "accuracy": 0.8256,
        },

        "majority_vote": {
            "qwk": 0.7917,
            "mae": 0.2093,
            "macro_f1": 0.6206,
            "accuracy": 0.8256,
        },

        "confidence_weighted_mean": {
            "qwk": 0.7409,
            "mae": 0.2442,
            "macro_f1": 0.6104,
            "accuracy": 0.8023,
        },

        "max_severity_prediction": {
            "qwk": 0.2487,
            "mae": 1.1395,
            "macro_f1": 0.3193,
            "accuracy": 0.3372,
        },
    },
}


for split, rule_values in (
    S1_VALUES.items()
):
    for aggregation_rule, metric_values in (
        rule_values.items()
    ):
        for metric_name, value in (
            metric_values.items()
        ):
            add_claim(
                claim_id=(
                    f"S1_{split}_{aggregation_rule}_"
                    f"{metric_name}"
                ),

                manuscript_location=(
                    "Supplementary Table S1, "
                    "pages 18–19"
                ),

                claim_kind=(
                    "aggregation_metric"
                ),

                manuscript_value=(
                    value
                ),

                split=(
                    split
                ),

                level=(
                    "patient"
                ),

                metric=(
                    metric_name
                ),

                aggregation_rule=(
                    aggregation_rule
                ),
            )


# ------------------------------------------------------------
# 17F. Patient and image confusion matrices
# ------------------------------------------------------------

MANUSCRIPT_PATIENT_CONFUSION = np.array(
    [
        [
            51,
            4,
            0,
            1,
        ],

        [
            4,
            15,
            0,
            0,
        ],

        [
            0,
            2,
            2,
            1,
        ],

        [
            0,
            0,
            3,
            3,
        ],
    ],
    dtype=int,
)


MANUSCRIPT_IMAGE_CONFUSION = np.array(
    [
        [
            822,
            101,
            2,
            0,
        ],

        [
            102,
            340,
            20,
            2,
        ],

        [
            3,
            29,
            119,
            26,
        ],

        [
            0,
            1,
            38,
            81,
        ],
    ],
    dtype=int,
)


for level, matrix in [
    (
        "patient",
        MANUSCRIPT_PATIENT_CONFUSION,
    ),

    (
        "image",
        MANUSCRIPT_IMAGE_CONFUSION,
    ),
]:
    for true_class in CLASS_LABELS:
        for predicted_class in CLASS_LABELS:
            add_claim(
                claim_id=(
                    f"CONFUSION_A3_test_{level}_"
                    f"t{true_class}_p{predicted_class}"
                ),

                manuscript_location=(
                    "Figure 2 / Supplementary "
                    "Figure S1, pages 11 and 17"
                ),

                claim_kind=(
                    "confusion_count"
                ),

                manuscript_value=int(
                    matrix[
                        true_class,
                        predicted_class,
                    ]
                ),

                reported_decimals=0,

                experiment_key=(
                    "A3_no_class_weights"
                ),

                split=(
                    "test"
                ),

                level=(
                    level
                ),

                true_class=(
                    true_class
                ),

                predicted_class=(
                    predicted_class
                ),
            )


# Image recalls stated in Section 4.7.
MANUSCRIPT_IMAGE_RECALL_PERCENT = {
    0: 88.9,
    1: 73.3,
    2: 67.2,
    3: 67.5,
}


for class_index, value in (
    MANUSCRIPT_IMAGE_RECALL_PERCENT.items()
):
    add_claim(
        claim_id=(
            f"NARRATIVE_A3_test_image_"
            f"class{class_index}_recall_percent"
        ),

        manuscript_location=(
            "Section 4.7, page 11"
        ),

        claim_kind=(
            "class_metric_percent"
        ),

        manuscript_value=(
            value
        ),

        reported_decimals=1,

        experiment_key=(
            "A3_no_class_weights"
        ),

        split=(
            "test"
        ),

        level=(
            "image"
        ),

        metric=(
            "recall"
        ),

        class_index=(
            class_index
        ),
    )


# ------------------------------------------------------------
# 17G. Legacy fold counts
# ------------------------------------------------------------

MANUSCRIPT_FOLD_PATIENT_COUNTS = [
    92,
    92,
    94,
    102,
    98,
]


for fold, patient_count in enumerate(
    MANUSCRIPT_FOLD_PATIENT_COUNTS
):
    add_claim(
        claim_id=(
            f"METHODS_LEGACY_FOLD_{fold}_PATIENTS"
        ),

        manuscript_location=(
            "Section 3.3, page 5"
        ),

        claim_kind=(
            "fold_patient_count"
        ),

        manuscript_value=(
            patient_count
        ),

        reported_decimals=0,

        experiment_key=(
            "A3_no_class_weights"
        ),

        fold=(
            fold
        ),
    )


# ------------------------------------------------------------
# 17H. Configuration claims
# ------------------------------------------------------------

CONFIG_CLAIMS = [
    (
        "A3_no_class_weights",
        "image_size",
        384,
        0,
        "Section 3.4",
    ),

    (
        "A3_no_class_weights",
        "batch_size",
        8,
        0,
        "Section 3.4",
    ),

    (
        "A3_no_class_weights",
        "epochs",
        25,
        0,
        "Section 3.8",
    ),

    (
        "A3_no_class_weights",
        "lr",
        1e-4,
        8,
        "Section 3.8",
    ),

    (
        "A3_no_class_weights",
        "min_lr_ratio",
        0.05,
        4,
        "Section 3.8",
    ),

    (
        "A3_no_class_weights",
        "warmup_epochs",
        3,
        0,
        "Section 3.8",
    ),

    (
        "A3_no_class_weights",
        "weight_decay",
        1e-4,
        8,
        "Section 3.8",
    ),

    (
        "A3_no_class_weights",
        "label_smoothing",
        0.02,
        4,
        "Section 3.8",
    ),

    (
        "A3_no_class_weights",
        "grad_clip_norm",
        1.0,
        2,
        "Section 3.8",
    ),

    (
        "A3_no_class_weights",
        "early_stop_patience",
        8,
        0,
        "Section 3.8",
    ),

    (
        "A3_no_class_weights",
        "ordinal_alpha",
        2.0,
        2,
        "Section 3.6",
    ),

    (
        "A3_no_class_weights",
        "ordinal_loss_weight",
        0.10,
        2,
        "Section 3.6",
    ),

    (
        "A3_no_class_weights",
        "use_class_weights_in_ce",
        False,
        None,
        "Sections 3.4 and 3.7",
    ),

    (
        "A3_no_class_weights",
        "use_tta",
        True,
        None,
        "Sections 3.4 and 3.9",
    ),
]


for (
    experiment_key,
    parameter,
    value,
    decimals,
    location,
) in CONFIG_CLAIMS:
    add_claim(
        claim_id=(
            f"CONFIG_{experiment_key}_{parameter}"
        ),

        manuscript_location=(
            location
        ),

        claim_kind=(
            "config_value"
        ),

        manuscript_value=(
            value
        ),

        reported_decimals=(
            decimals
        ),

        experiment_key=(
            experiment_key
        ),

        config_parameter=(
            parameter
        ),
    )


claim_registry_df = pd.DataFrame(
    claim_rows
)


atomic_csv_save(
    claim_registry_df,

    OUTPUT_ROOT
    / "legacy_claim_registry.csv",

    index=False,
)


# ============================================================
# 18. Configuration lookup helpers
# ============================================================

CONFIG_PARAMETER_ALIASES = {
    "image_size": [
        "image_size",
    ],

    "batch_size": [
        "batch_size",
        "train_batch_size",
    ],

    "epochs": [
        "epochs",
        "num_epochs",
    ],

    "lr": [
        "lr",
        "learning_rate",
    ],

    "min_lr_ratio": [
        "min_lr_ratio",
    ],

    "warmup_epochs": [
        "warmup_epochs",
        "num_warmup_epochs",
    ],

    "weight_decay": [
        "weight_decay",
    ],

    "label_smoothing": [
        "label_smoothing",
    ],

    "grad_clip_norm": [
        "grad_clip_norm",
    ],

    "early_stop_patience": [
        "early_stop_patience",
        "patience",
    ],

    "ordinal_alpha": [
        "ordinal_alpha",
        "alpha",
    ],

    "ordinal_loss_weight": [
        "ordinal_loss_weight",
        "ordinal_weight",
        "lambda",
    ],

    "use_class_weights_in_ce": [
        "use_class_weights_in_ce",
        "use_class_weights",
    ],

    "use_tta": [
        "use_tta",
    ],
}


def find_config_value(
    experiment_key,
    parameter,
):
    if experiment_key not in config_lookup:
        return None, None

    aliases = CONFIG_PARAMETER_ALIASES.get(
        parameter,
        [
            parameter,
        ],
    )

    flattened = config_lookup[
        experiment_key
    ]

    for alias in aliases:
        normalized_alias = normalize_name(
            alias
        )

        for parameter_path, value in (
            flattened.items()
        ):
            leaf = normalize_name(
                parameter_path.split(
                    "."
                )[
                    -1
                ]
            )

            if leaf == normalized_alias:
                return value, parameter_path

    return None, None


# ============================================================
# 19. Compare manuscript claims with recomputed values
# ============================================================

comparison_rows = []


def values_match_at_precision(
    manuscript_value,
    computed_value,
    decimals,
):
    if (
        manuscript_value is None
        or
        (
            isinstance(
                manuscript_value,
                float,
            )
            and
            np.isnan(
                manuscript_value
            )
        )
    ):
        return None

    if isinstance(
        manuscript_value,
        (
            bool,
            np.bool_,
        ),
    ):
        return bool(
            manuscript_value
        ) == bool(
            computed_value
        )

    if isinstance(
        manuscript_value,
        str,
    ):
        return (
            str(
                manuscript_value
            ).strip()
            ==
            str(
                computed_value
            ).strip()
        )

    if decimals is None:
        return (
            manuscript_value
            ==
            computed_value
        )

    tolerance = (
        0.5
        *
        (
            10.0
            **
            (
                -int(
                    decimals
                )
            )
        )
        +
        1e-12
    )

    return (
        abs(
            float(
                manuscript_value
            )
            -
            float(
                computed_value
            )
        )
        <=
        tolerance
    )


for _, claim in claim_registry_df.iterrows():
    claim_kind = claim[
        "claim_kind"
    ]

    computed_value = None
    source_type = None
    source_path = None

    if claim_kind == "metric":
        lookup_key = (
            claim[
                "experiment_key"
            ],
            claim[
                "split"
            ],
            claim[
                "level"
            ],
            claim[
                "metric"
            ],
        )

        lookup_record = metric_lookup.get(
            lookup_key
        )

        if lookup_record is not None:
            computed_value = (
                lookup_record[
                    "value"
                ]
            )

            source_type = (
                lookup_record[
                    "source_type"
                ]
            )

            source_path = (
                lookup_record[
                    "source_path"
                ]
            )

    elif claim_kind == "aggregation_metric":
        lookup_key = (
            claim[
                "split"
            ],
            claim[
                "aggregation_rule"
            ],
            claim[
                "metric"
            ],
        )

        lookup_record = (
            aggregation_lookup.get(
                lookup_key
            )
        )

        if lookup_record is not None:
            computed_value = (
                lookup_record[
                    "value"
                ]
            )

            source_type = (
                lookup_record[
                    "source_type"
                ]
            )

            source_path = (
                lookup_record[
                    "source_path"
                ]
            )

    elif claim_kind == "confusion_count":
        lookup_key = (
            claim[
                "experiment_key"
            ],
            claim[
                "split"
            ],
            claim[
                "level"
            ],
            int(
                claim[
                    "true_class"
                ]
            ),
            int(
                claim[
                    "predicted_class"
                ]
            ),
        )

        lookup_record = (
            confusion_lookup.get(
                lookup_key
            )
        )

        if lookup_record is not None:
            computed_value = (
                lookup_record[
                    "value"
                ]
            )

            source_type = (
                "independent_confusion_recomputation"
            )

            source_path = (
                lookup_record[
                    "source_path"
                ]
            )

    elif claim_kind == "class_metric_percent":
        lookup_key = (
            claim[
                "experiment_key"
            ],
            claim[
                "split"
            ],
            claim[
                "level"
            ],
            int(
                claim[
                    "class_index"
                ]
            ),
            claim[
                "metric"
            ],
        )

        lookup_record = (
            class_metric_lookup.get(
                lookup_key
            )
        )

        if lookup_record is not None:
            computed_value = (
                100.0
                *
                float(
                    lookup_record[
                        "value"
                    ]
                )
            )

            source_type = (
                "independent_per_class_recomputation"
            )

            source_path = (
                lookup_record[
                    "source_path"
                ]
            )

    elif claim_kind == "fold_patient_count":
        fold = int(
            claim[
                "fold"
            ]
        )

        lookup_record = (
            fold_count_lookup.get(
                fold
            )
        )

        if lookup_record is not None:
            computed_value = (
                lookup_record[
                    "value"
                ]
            )

            source_type = (
                "unique_patients_in_saved_fold_predictions"
            )

            source_path = (
                lookup_record[
                    "source_path"
                ]
            )

    elif claim_kind == "config_value":
        (
            computed_value,
            config_parameter_path,
        ) = find_config_value(
            experiment_key=(
                claim[
                    "experiment_key"
                ]
            ),

            parameter=(
                claim[
                    "config_parameter"
                ]
            ),
        )

        if computed_value is not None:
            source_type = (
                "legacy_config_json"
            )

            source_path = (
                str(
                    selected_config_files.get(
                        claim[
                            "experiment_key"
                        ],
                        "",
                    )
                )
                +
                (
                    f"::{config_parameter_path}"
                    if config_parameter_path
                    is not None
                    else
                    ""
                )
            )

    manuscript_value = claim[
        "manuscript_value"
    ]

    manuscript_value_is_missing = (
        pd.isna(
            manuscript_value
        )
    )

    computed_value_is_missing = (
        computed_value is None
        or
        (
            isinstance(
                computed_value,
                float,
            )
            and
            np.isnan(
                computed_value
            )
        )
    )

    if (
        manuscript_value_is_missing
        and
        not computed_value_is_missing
    ):
        status = (
            "manuscript_omitted_value_recovered"
        )

        match = None

    elif (
        manuscript_value_is_missing
        and
        computed_value_is_missing
    ):
        status = (
            "manuscript_omitted_value_not_recovered"
        )

        match = None

    elif computed_value_is_missing:
        status = (
            "not_recomputed"
        )

        match = None

    else:
        match = values_match_at_precision(
            manuscript_value=(
                manuscript_value
            ),

            computed_value=(
                computed_value
            ),

            decimals=(
                claim[
                    "reported_decimals"
                ]
            ),
        )

        status = (
            "match_at_reported_precision"
            if match
            else
            "mismatch"
        )

    numeric_difference = np.nan

    if (
        not manuscript_value_is_missing
        and
        not computed_value_is_missing
        and
        isinstance(
            manuscript_value,
            (
                int,
                float,
                np.number,
            ),
        )
        and
        isinstance(
            computed_value,
            (
                int,
                float,
                np.number,
            ),
        )
    ):
        numeric_difference = (
            float(
                computed_value
            )
            -
            float(
                manuscript_value
            )
        )

    comparison_rows.append({
        **claim.to_dict(),

        "computed_value": (
            computed_value
        ),

        "computed_minus_manuscript": (
            numeric_difference
        ),

        "comparison_status": (
            status
        ),

        "match_at_reported_precision": (
            match
        ),

        "computed_source_type": (
            source_type
        ),

        "computed_source_path": (
            source_path
        ),

        "revision_status": (
            "superseded_by_route_A"
        ),
    })


manuscript_comparison_df = pd.DataFrame(
    comparison_rows
)


atomic_csv_save(
    manuscript_comparison_df,

    OUTPUT_ROOT
    / "legacy_manuscript_value_comparison.csv",

    index=False,
)


DISCREPANCY_STATUSES = {
    "mismatch",
    "not_recomputed",
    "manuscript_omitted_value_recovered",
    "manuscript_omitted_value_not_recovered",
}


discrepancy_df = (
    manuscript_comparison_df[
        manuscript_comparison_df[
            "comparison_status"
        ].isin(
            DISCREPANCY_STATUSES
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


atomic_csv_save(
    discrepancy_df,

    OUTPUT_ROOT
    / "legacy_discrepancy_report.csv",

    index=False,
)


# ============================================================
# 20. Inferential-validity audit
# ============================================================

inferential_validity_rows = [
    {
        "issue_id": (
            "IV01_test_use_contradiction"
        ),

        "manuscript_location": (
            "Sections 3.3 and 3.7"
        ),

        "legacy_evidence": (
            "Section 3.3 states that the reserved test "
            "set was not used for ablation selection, "
            "whereas Section 3.7 selects A3 because test "
            "patient QWK increased from 0.7946 to 0.8213."
        ),

        "computational_correctness_question": (
            "The numerical values may be correct outputs "
            "of the legacy runs."
        ),

        "inferential_assessment": (
            "The test result cannot remain an untouched "
            "confirmatory estimate after it influenced "
            "ablation selection."
        ),

        "route_A_resolution": (
            "RA01–RA07 reconstructed selection using "
            "development data; RA08 locked the system; "
            "RA09 performed a locked re-evaluation."
        ),

        "residual_limitation": (
            "Historical investigator exposure to the "
            "official test set cannot be erased."
        ),
    },

    {
        "issue_id": (
            "IV02_backbone_selection"
        ),

        "manuscript_location": (
            "Section 4.3 and Table 2"
        ),

        "legacy_evidence": (
            "ConvNeXt had the highest OOF patient QWK "
            "(0.8508), but Swin was advanced because its "
            "test patient QWK was higher."
        ),

        "computational_correctness_question": (
            "Audit whether Table 2 values match saved "
            "prediction files."
        ),

        "inferential_assessment": (
            "Backbone selection appears influenced by "
            "official-test performance."
        ),

        "route_A_resolution": (
            "RA06 performed a uniform development-only "
            "ViT–Swin comparison."
        ),

        "residual_limitation": (
            "ConvNeXt and MaxViT were removed from Route A."
        ),
    },

    {
        "issue_id": (
            "IV03_aggregation_selection"
        ),

        "manuscript_location": (
            "Section 4.6 and Supplementary Table S1"
        ),

        "legacy_evidence": (
            "Median probability was best OOF, while mean "
            "probability was retained partly because it "
            "was best on the test set."
        ),

        "computational_correctness_question": (
            "Audit the saved aggregation table and "
            "independently recomputable rules."
        ),

        "inferential_assessment": (
            "Primary aggregation selection was influenced "
            "by official-test performance."
        ),

        "route_A_resolution": (
            "RA03 selected mean probability from "
            "development OOF predictions before RA08."
        ),

        "residual_limitation": (
            "RA09 test sensitivity remains descriptive only."
        ),
    },

    {
        "issue_id": (
            "IV04_missing_uncertainty"
        ),

        "manuscript_location": (
            "Abstract, Tables 2–4, Sections 4–5"
        ),

        "legacy_evidence": (
            "Only point estimates were reported for "
            "86 test patients."
        ),

        "computational_correctness_question": (
            "Not a transcription problem."
        ),

        "inferential_assessment": (
            "Small model differences and severe-class "
            "estimates cannot be interpreted reliably "
            "without uncertainty."
        ),

        "route_A_resolution": (
            "RA02, RA05–RA07, and RA09 added multi-seed "
            "results and patient-bootstrap confidence "
            "intervals."
        ),

        "residual_limitation": (
            "Bootstrap intervals condition on trained models."
        ),
    },

    {
        "issue_id": (
            "IV05_patient_reference_definition"
        ),

        "manuscript_location": (
            "Sections 3.2, 3.10, 4.6, and Discussion"
        ),

        "legacy_evidence": (
            "Prediction aggregation was defined, but the "
            "construction of one patient true label from "
            "heterogeneous image labels was not clearly "
            "specified."
        ),

        "computational_correctness_question": (
            "Legacy code appears to use a low-tie modal "
            "rule, which should be verified from code."
        ),

        "inferential_assessment": (
            "Patient-level metrics are inseparable from "
            "the reference construction."
        ),

        "route_A_resolution": (
            "RA01 audited modal_low, modal_high, median, "
            "and maximum references; RA09 quantified test "
            "sensitivity."
        ),

        "residual_limitation": (
            "No independently adjudicated examination-level "
            "MES exists."
        ),
    },

    {
        "issue_id": (
            "IV06_small_severe_groups"
        ),

        "manuscript_location": (
            "Sections 4.7 and 5"
        ),

        "legacy_evidence": (
            "Patient results used only five Mayo 2 and six "
            "Mayo 3 patients."
        ),

        "computational_correctness_question": (
            "Confusion counts can be recomputed."
        ),

        "inferential_assessment": (
            "Per-class percentages and F1 values are highly "
            "unstable with these denominators."
        ),

        "route_A_resolution": (
            "RA09 reports exact supports and bootstrap CIs."
        ),

        "residual_limitation": (
            "The sample size remains five and six patients."
        ),
    },

    {
        "issue_id": (
            "IV07_overstated_clinical_claims"
        ),

        "manuscript_location": (
            "Abstract, Introduction, Discussion, Conclusion"
        ),

        "legacy_evidence": (
            "Terms such as clinically aligned and most "
            "clinically informative were used without "
            "external or specialist validation."
        ),

        "computational_correctness_question": (
            "Not a numerical issue."
        ),

        "inferential_assessment": (
            "Grouped LIMUC evaluation is not equivalent to "
            "clinical examination-level validation."
        ),

        "route_A_resolution": (
            "Reframe as evaluation-granularity and "
            "derived-reference benchmarking."
        ),

        "residual_limitation": (
            "No external clinical validation."
        ),
    },

    {
        "issue_id": (
            "IV08_missing_M1_M2_MAE"
        ),

        "manuscript_location": (
            "Table 2"
        ),

        "legacy_evidence": (
            "Test patient MAE was shown as an em dash for "
            "M1 and M2."
        ),

        "computational_correctness_question": (
            "Recover values from saved patient predictions "
            "when available."
        ),

        "inferential_assessment": (
            "The old comparison was incomplete."
        ),

        "route_A_resolution": (
            "Route A reports complete metrics for both "
            "primary models."
        ),

        "residual_limitation": (
            "Recovered legacy values are provenance only."
        ),
    },

    {
        "issue_id": (
            "IV09_gradcam_superseded_model"
        ),

        "manuscript_location": (
            "Figure 4 and Supplementary Figure S2"
        ),

        "legacy_evidence": (
            "Saliency figures were generated from the "
            "legacy A3 model."
        ),

        "computational_correctness_question": (
            "Exact checkpoint and source code should be "
            "identified from LG-CODE."
        ),

        "inferential_assessment": (
            "These figures cannot explain the revised "
            "locked class-weighted Swin ensemble."
        ),

        "route_A_resolution": (
            "Remove, move to provenance-only supplementary "
            "material, or regenerate with a locked Route A "
            "checkpoint."
        ),

        "residual_limitation": (
            "No IBD-specialist saliency validation."
        ),
    },
]


inferential_validity_df = pd.DataFrame(
    inferential_validity_rows
)


atomic_csv_save(
    inferential_validity_df,

    OUTPUT_ROOT
    / "legacy_inferential_validity_audit.csv",

    index=False,
)


# ============================================================
# 21. Hash source files actually used
# ============================================================

used_source_paths = set()


for path in selected_config_files.values():
    if path is not None:
        used_source_paths.add(
            Path(
                path
            )
        )


for experiment_files in (
    selected_prediction_files.values()
):
    for path in experiment_files.values():
        if path is not None:
            used_source_paths.add(
                Path(
                    path
                )
            )


if not aggregation_saved_summary_df.empty:
    for path in (
        aggregation_saved_summary_df[
            "source_path"
        ].dropna()
    ):
        used_source_paths.add(
            Path(
                path
            )
        )


for path in code_evidence_df.get(
    "source_path",
    pd.Series(
        dtype=str
    ),
).dropna():
    used_source_paths.add(
        Path(
            path
        )
    )


source_provenance_rows = []


for path in sorted(
    used_source_paths,
    key=lambda item: str(
        item
    ),
):
    if not path.exists():
        continue

    try:
        sha256 = file_sha256(
            path
        )

    except Exception:
        sha256 = None

    source_provenance_rows.append({
        "path": str(
            path
        ),

        "relative_to_legacy_root": (
            str(
                path.relative_to(
                    LEGACY_ROOT
                )
            )
            if LEGACY_ROOT in path.parents
            else
            ""
        ),

        "size_bytes": int(
            path.stat().st_size
        ),

        "sha256": (
            sha256
        ),

        "checkpoint_loaded": False,
    })


source_provenance_df = pd.DataFrame(
    source_provenance_rows
)


atomic_csv_save(
    source_provenance_df,

    OUTPUT_ROOT
    / "legacy_source_provenance_manifest.csv",

    index=False,
)


# ============================================================
# 22. Audit summary
# ============================================================

status_counts = (
    manuscript_comparison_df[
        "comparison_status"
    ]
    .value_counts(
        dropna=False
    )
    .to_dict()
)


matched_claims = int(
    status_counts.get(
        "match_at_reported_precision",
        0,
    )
)


mismatched_claims = int(
    status_counts.get(
        "mismatch",
        0,
    )
)


not_recomputed_claims = int(
    status_counts.get(
        "not_recomputed",
        0,
    )
)


recovered_omissions = int(
    status_counts.get(
        "manuscript_omitted_value_recovered",
        0,
    )
)


unrecovered_omissions = int(
    status_counts.get(
        "manuscript_omitted_value_not_recovered",
        0,
    )
)


provenance_manifest = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "notebook_name": (
        "00_legacy_manuscript_reproducibility_audit.ipynb"
    ),

    "purpose": (
        "Read-only audit of pre-Route-A saved outputs "
        "against the submitted manuscript."
    ),

    "legacy_root": str(
        LEGACY_ROOT
    ),

    "route_A_root": str(
        ROUTE_A_ROOT
    ),

    "output_root": str(
        OUTPUT_ROOT
    ),

    "raw_image_accessed": False,

    "checkpoints_loaded": False,

    "model_inference_performed": False,

    "training_performed": False,

    "manuscript_source": (
        "Previously submitted 19-page PDF; claims encoded "
        "from Abstract, Sections 3–5, Tables 2–4, "
        "Figures 2/S1, and Supplementary Table S1."
    ),

    "experiment_directories": {
        experiment_key: str(
            experiment_info[
                "path"
            ]
        )

        for experiment_key, experiment_info in (
            LEGACY_EXPERIMENTS.items()
        )
    },

    "aggregation_root": str(
        AGGREGATION_ROOT
    ),

    "code_root": str(
        CODE_ROOT
    ),

    "claim_count": int(
        len(
            claim_registry_df
        )
    ),

    "comparison_status_counts": {
        str(
            key
        ): int(
            value
        )

        for key, value in (
            status_counts.items()
        )
    },

    "matched_claims": (
        matched_claims
    ),

    "mismatched_claims": (
        mismatched_claims
    ),

    "not_recomputed_claims": (
        not_recomputed_claims
    ),

    "recovered_manuscript_omissions": (
        recovered_omissions
    ),

    "unrecovered_manuscript_omissions": (
        unrecovered_omissions
    ),

    "trim_fraction_inferred_from_code": (
        INFERRED_TRIM_FRACTION
    ),

    "interpretation": (
        "Numerical agreement confirms only that a legacy "
        "value was reproduced from preserved outputs. It "
        "does not restore inferential validity when the "
        "official test set influenced selection."
    ),

    "authoritative_revision_pipeline": (
        "Route A Notebooks 01–09"
    ),

    "environment": {
        "python": (
            platform.python_version()
        ),

        "platform": (
            platform.platform()
        ),

        "numpy": (
            np.__version__
        ),

        "pandas": (
            pd.__version__
        ),

        "scikit_learn": (
            safe_package_version(
                "scikit-learn"
            )
        ),

        "scipy": (
            safe_package_version(
                "scipy"
            )
        ),
    },
}


atomic_json_save(
    provenance_manifest,

    OUTPUT_ROOT
    / "legacy_provenance_manifest.json",
)


summary_text = f"""
LEGACY MANUSCRIPT REPRODUCIBILITY AUDIT
=======================================

Created UTC:
{provenance_manifest['created_utc']}

Legacy root:
{LEGACY_ROOT}

Output root:
{OUTPUT_ROOT}

Training performed:
NO

Model checkpoints loaded:
NO

Image inference performed:
NO

Raw development/test images accessed:
NO

Total manuscript claims registered:
{len(claim_registry_df)}

Claims matching at reported precision:
{matched_claims}

Claims with numerical mismatch:
{mismatched_claims}

Claims not independently recomputed:
{not_recomputed_claims}

Manuscript omissions recovered from saved outputs:
{recovered_omissions}

Manuscript omissions not recovered:
{unrecovered_omissions}

Inferred legacy trimmed-mean fraction:
{INFERRED_TRIM_FRACTION}

IMPORTANT INTERPRETATION
------------------------
A matching legacy value means that the saved legacy output
reproduces the number reported in the submitted manuscript.

It does NOT mean that the original model-selection logic was
confirmatory or that the official test estimate remained
untouched after test-guided selection.

All current revised manuscript results must come from
Route A Notebooks 01–09.

The legacy analyses are retained only for:
- provenance;
- correction of transcription errors;
- recovery of omitted metrics;
- documentation of historical test exposure; and
- explanation of why the analysis was reconstructed.
"""


atomic_text_save(
    summary_text.strip()
    +
    "\n",

    OUTPUT_ROOT
    / "LEGACY_AUDIT_SUMMARY.txt",
)


# ============================================================
# 23. Display results
# ============================================================

print("\n" + "=" * 125)
print("LEGACY EXPERIMENT DIRECTORY STATUS")
print("=" * 125)


directory_status_df = pd.DataFrame([
    {
        "experiment_key": (
            experiment_key
        ),

        "display_name": (
            experiment_info[
                "display_name"
            ]
        ),

        "path": str(
            experiment_info[
                "path"
            ]
        ),

        "exists": bool(
            experiment_info[
                "path"
            ].exists()
        ),
    }

    for experiment_key, experiment_info in (
        LEGACY_EXPERIMENTS.items()
    )
])


display(
    directory_status_df
)


print("\n" + "=" * 125)
print("MANUSCRIPT CLAIM COMPARISON STATUS")
print("=" * 125)


status_summary_df = (
    manuscript_comparison_df[
        "comparison_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "comparison_status"
    )
    .reset_index(
        name="n_claims"
    )
)


display(
    status_summary_df
)


print("\n" + "=" * 125)
print("RECOVERED MANUSCRIPT OMISSIONS")
print("=" * 125)


recovered_df = (
    manuscript_comparison_df[
        manuscript_comparison_df[
            "comparison_status"
        ]
        ==
        "manuscript_omitted_value_recovered"
    ][
        [
            "claim_id",
            "manuscript_location",
            "experiment_key",
            "split",
            "level",
            "metric",
            "computed_value",
            "computed_source_path",
        ]
    ]
)


if recovered_df.empty:
    print(
        "No omitted manuscript values were recovered."
    )

else:
    display(
        recovered_df
    )


print("\n" + "=" * 125)
print("NUMERICAL MISMATCHES")
print("=" * 125)


mismatch_df = (
    manuscript_comparison_df[
        manuscript_comparison_df[
            "comparison_status"
        ]
        ==
        "mismatch"
    ][
        [
            "claim_id",
            "manuscript_location",
            "manuscript_value",
            "computed_value",
            "computed_minus_manuscript",
            "computed_source_type",
            "computed_source_path",
        ]
    ]
)


if mismatch_df.empty:
    print(
        "No numerical mismatches were found among "
        "independently recomputed claims."
    )

else:
    display(
        mismatch_df
    )


print("\n" + "=" * 125)
print("CLAIMS NOT INDEPENDENTLY RECOMPUTED")
print("=" * 125)


not_recomputed_df = (
    manuscript_comparison_df[
        manuscript_comparison_df[
            "comparison_status"
        ]
        ==
        "not_recomputed"
    ][
        [
            "claim_id",
            "manuscript_location",
            "claim_kind",
            "experiment_key",
            "split",
            "level",
            "metric",
            "aggregation_rule",
            "manuscript_value",
            "note",
        ]
    ]
)


if not_recomputed_df.empty:
    print(
        "All registered numerical claims were recomputed."
    )

else:
    display(
        not_recomputed_df
    )


print("\n" + "=" * 125)
print("LEGACY PATIENT-PREDICTION CONSISTENCY")
print("=" * 125)


if patient_consistency_df.empty:
    print(
        "No explicit-vs-derived patient prediction "
        "comparisons were available."
    )

else:
    display(
        patient_consistency_df
    )


print("\n" + "=" * 125)
print("LEGACY FOLD PATIENT COUNTS")
print("=" * 125)


if fold_count_df.empty:
    print(
        "Fold patient counts could not be recovered."
    )

else:
    display(
        fold_count_df
    )


print("\n" + "=" * 125)
print("LEGACY AGGREGATION SUMMARY — SAVED OUTPUT")
print("=" * 125)


if aggregation_saved_summary_df.empty:
    print(
        "No machine-readable aggregation summary "
        "was identified."
    )

else:
    display(
        aggregation_saved_summary_df
    )


print("\n" + "=" * 125)
print("LEGACY AGGREGATION — INDEPENDENT RECOMPUTATION")
print("=" * 125)


if aggregation_recomputed_df.empty:
    print(
        "Aggregation could not be independently recomputed."
    )

else:
    display(
        aggregation_recomputed_df
    )


print("\n" + "=" * 125)
print("INFERENTIAL-VALIDITY AUDIT")
print("=" * 125)


display(
    inferential_validity_df
)


# ============================================================
# 24. Final status
# ============================================================

print("\n" + "=" * 125)
print("00_legacy_manuscript_reproducibility_audit COMPLETED")
print("=" * 125)


print(
    "\nOutput directory:"
)

print(
    OUTPUT_ROOT
)


print(
    "\nPrimary comparison file:"
)

print(
    OUTPUT_ROOT
    / "legacy_manuscript_value_comparison.csv"
)


print(
    "\nDiscrepancy report:"
)

print(
    OUTPUT_ROOT
    / "legacy_discrepancy_report.csv"
)


print(
    "\nInferential-validity audit:"
)

print(
    OUTPUT_ROOT
    / "legacy_inferential_validity_audit.csv"
)


print(
    "\nProvenance manifest:"
)

print(
    OUTPUT_ROOT
    / "legacy_provenance_manifest.json"
)


print(
    "\nAudit summary:"
)

print(
    OUTPUT_ROOT
    / "LEGACY_AUDIT_SUMMARY.txt"
)


print(
    "\nIMPORTANT:"
)

print(
    "A reproduced legacy value confirms numerical "
    "provenance only."
)

print(
    "It does not make a test-guided selection pathway "
    "confirmatory."
)

print(
    "All revised manuscript performance claims must "
    "continue to use Route A Notebooks 01–09."
)